## collect game data

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import re
import pandas as pd

# Setup headless browser
options = Options()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
driver = webdriver.Chrome(options=options)

try:
    # Base URL
    url = "https://www.koreabaseball.com/Schedule/Schedule.aspx"
    
    # Open site
    driver.get(url)
    wait = WebDriverWait(driver, 10)
    
    # Ensure dropdowns load
    wait.until(EC.presence_of_element_located((By.ID, "ddlYear")))
    wait.until(EC.presence_of_element_located((By.ID, "ddlMonth")))
    
    # Select 2024
    year_dropdown = driver.find_element(By.ID, "ddlYear")
    year_dropdown.find_element(By.XPATH, "./option[@value='2024']").click()
    time.sleep(2)  # Give time for any page updates
    
    # First, let's check the actual values in the month dropdown
    month_dropdown = driver.find_element(By.ID, "ddlMonth")
    month_options = month_dropdown.find_elements(By.TAG_NAME, "option")
    month_values = {option.get_attribute('value'): option.text for option in month_options}
    print("Available month values:", month_values)
    
    results = []
    
    # Focus on months that have games
    active_months = ['03', '04', '05', '06', '07', '08', '09', '10']
    
    for month_value in active_months:
        print(f"Processing month: {month_values.get(month_value, month_value)} (value: {month_value})")
        
        # Select month using the actual value
        month_dropdown = driver.find_element(By.ID, "ddlMonth")
        month_dropdown.find_element(By.XPATH, f"./option[@value='{month_value}']").click()
        time.sleep(2)
    
        # Trigger table refresh
        driver.execute_script("getTableGridList();")
        time.sleep(3)  # Longer wait to ensure page loads
    
        # Parse updated page
        soup = BeautifulSoup(driver.page_source, "html.parser")
        table = soup.find("table", {"id": "tblScheduleList"})
        if not table:
            print(f"No table found for month {month_value}")
            continue
    
        tbody = table.find("tbody")
        if not tbody:
            print(f"No tbody found for month {month_value}")
            continue
            
        rows = tbody.find_all("tr")
        print(f"Found {len(rows)} rows for month {month_value}")
        
        current_date = None
        
        for i, row in enumerate(rows):
            # Print the HTML of the first few rows for debugging
            if i < 5:
                print(f"Row {i} HTML: {row}")
            
            # Check for date in both th.day and td.day
            date_cell = row.find("th", class_="day") or row.find("td", class_="day")
            if date_cell and date_cell.text.strip():
                current_date = date_cell.text.strip()
                print(f"Found date: {current_date}")
            
            # Extract game info
            play_cell = row.find("td", class_="play")
            if play_cell:
                match_text = play_cell.text.strip()
                
                # Get stadium info
                stadium_td = row.find_all("td")
                stadium = ""
                if len(stadium_td) > 7:
                    stadium = stadium_td[7].text.strip()
                
                # Skip if no match text
                if not match_text:
                    continue
                    
                print(f"Processing match text: {match_text}")
                
                # Extract teams and scores using span elements
                away_team = home_team = away_score = home_score = None
                
                # Get scores from win/lose spans
                score_spans = play_cell.find_all("span", class_=["win", "lose"])
                if len(score_spans) >= 2:
                    away_score = int(score_spans[0].text.strip()) if score_spans[0].text.strip().isdigit() else None
                    home_score = int(score_spans[1].text.strip()) if score_spans[1].text.strip().isdigit() else None
                
                # Get team names from regular spans (excluding vs, win, lose spans)
                team_spans = []
                for span in play_cell.find_all("span"):
                    if "vs" not in span.text and not span.has_attr("class"):
                        team_spans.append(span.text.strip())
                    elif span.has_attr("class") and not any(cls in ["win", "lose"] for cls in span.get("class", [])):
                        team_spans.append(span.text.strip())
                
                if len(team_spans) >= 2:
                    away_team = team_spans[0]
                    home_team = team_spans[-1]  # Last team is home team
                else:
                    # Fallback to regex if HTML parsing fails
                    vs_parts = match_text.split("vs")
                    if len(vs_parts) == 2:
                        # Clean up any digits that might be scores
                        away_part = vs_parts[0].strip()
                        home_part = vs_parts[1].strip()
                        
                        # Extract away team and score
                        away_score_match = re.search(r'(.+?)(\d+)$', away_part)
                        if away_score_match:
                            away_team = away_score_match.group(1).strip()
                            if away_score is None:  # Don't override if we already got from spans
                                away_score = int(away_score_match.group(2))
                        else:
                            away_team = away_part
                        
                        # Extract home team and score
                        home_score_match = re.match(r'^(\d+)(.+)$', home_part)
                        if home_score_match:
                            home_score = int(home_score_match.group(1)) if home_score is None else home_score
                            home_team = home_score_match.group(2).strip()
                        else:
                            home_team = home_part
                
                if current_date and away_team and home_team:
                    results.append({
                        "date": current_date,
                        "home_team": home_team.strip(),
                        "away_team": away_team.strip(),
                        "home_score": home_score,
                        "away_score": away_score,
                        "stadium": stadium
                    })
                    print(f"Added game: {away_team} ({away_score}) vs {home_team} ({home_score}) on {current_date}")
                
            # Also process regular rows with no td.play
            elif not play_cell:
                cols = row.find_all("td")
                if not cols or len(cols) < 2:
                    continue
                
                # Check for date in time column
                time_str = cols[0].text.strip() if len(cols) > 0 else ""
                if re.match(r'\d{2}\.\d{2}\(\w+\)', time_str):
                    current_date = time_str
                    print(f"Found date in time column: {current_date}")
                    continue
                
                # Process regular game info
                match_text = cols[1].text.strip() if len(cols) > 1 else ""
                stadium = cols[6].text.strip() if len(cols) > 6 else ""
                
                if not match_text:
                    continue
                
                print(f"Processing regular match text: {match_text}")
                
                # Process the match text
                score_match = re.search(r"(.+?)\s*(\d+)\s*:\s*(\d+)\s*(.+)", match_text)
                if score_match:
                    away_team, away_score, home_score, home_team = score_match.groups()
                    away_score, home_score = int(away_score), int(home_score)
                else:
                    teams = match_text.split("vs")
                    if len(teams) != 2:
                        print(f"Unexpected match format: {match_text}")
                        continue
                        
                    away_team = teams[0].strip()
                    home_team = teams[1].strip()
                    away_score = home_score = None
                    
                    # Extract scores if they're in team names
                    away_score_match = re.search(r'(.+?)(\d+)$', away_team)
                    if away_score_match:
                        away_team = away_score_match.group(1).strip()
                        away_score = int(away_score_match.group(2))
                    
                    home_score_match = re.match(r'^(\d+)(.+)$', home_team)
                    if home_score_match:
                        home_score = int(home_score_match.group(1))
                        home_team = home_score_match.group(2).strip()
                
                if current_date:
                    results.append({
                        "date": current_date,
                        "home_team": home_team.strip(),
                        "away_team": away_team.strip(),
                        "home_score": home_score,
                        "away_score": away_score,
                        "stadium": stadium
                    })
                    print(f"Added game: {away_team} ({away_score}) vs {home_team} ({home_score}) on {current_date}")
    
    # Print summary
    print(f"Total games collected: {len(results)}")
    
    if not results:
        print("WARNING: No data was collected!")
    else:
        # Save to CSV without time column as requested
        df = pd.DataFrame(results)
        df.to_csv("kbo_2024_schedule.csv", index=False, encoding="utf-8-sig")
        print("Data saved to kbo_2024_schedule.csv")
        print(df.head())

except Exception as e:
    import traceback
    print(f"An error occurred: {str(e)}")
    print(traceback.format_exc())

finally:
    # Ensure browser is closed
    driver.quit()

## fixed the column

In [ ]:
import pandas as pd

# Load the existing schedule
df = pd.read_csv("kbo_2024_schedule.csv")

print(f"Original data shape: {df.shape}")

# Drop games that contain '우천취소' (rain cancellation) in any column
rain_cancelled = df.apply(lambda row: any('우천취소' in str(val) for val in row.values), axis=1)
df_clean = df[~rain_cancelled]

print(f"After removing rain-cancelled games: {df_clean.shape}")

# Drop the stadium column
if 'stadium' in df_clean.columns:
    df_clean = df_clean.drop('stadium', axis=1)
    print("Removed stadium column")

# Save the cleaned file
df_clean.to_csv("kbo_2024_schedule_clean.csv", index=False, encoding="utf-8-sig")
print("Cleaned data saved to kbo_2024_schedule_clean.csv")
print(df_clean.head())

## extract player's performance

In [ ]:
import requests
import json

payload = {
    'leId': '1',
    'srId': '0,1,3,4,5,7,9',
    'date': '20240323'
}

res = requests.post(
    'https://www.koreabaseball.com/ws/Main.asmx/GetKboGameList',
    json=payload,
    headers={'User-Agent': 'Mozilla/5.0'}
)

raw = res.text.strip()

#  Decode only the first valid JSON object from the string
decoder = json.JSONDecoder()
data, _ = decoder.raw_decode(raw)

# 🎯 Now access your data safely
games = data['game']
for g in games:
    print(g['G_ID'], g['S_NM'], g['AWAY_NM'], 'vs', g['HOME_NM'], g['T_SCORE_CN'], '-', g['B_SCORE_CN'])


In [ ]:
import json
import pandas as pd

# Load your saved Postman response
with open("response.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Parse arrHitter
def parse_arr_hitter(data, meta):
    all_rows = []

    for team_data in data['arrHitter']:
        table1 = json.loads(team_data['table1'])  # player info
        table3 = json.loads(team_data['table3'])  # performance stats

        for i, row in enumerate(table1['rows']):
            try:
                cols = row['row']
                order = cols[0]['Text'].strip()
                pos = cols[1]['Text'].strip()
                name = cols[2]['Text'].strip()

                perf = [td['Text'].strip() for td in table3['rows'][i]['row']]
                # Example: ['4', '2', '1', '1', '0.308'] → AB, H, RBI, R, AVG

                all_rows.append([
                    meta['leId'], meta['srId'], meta['seasonId'], meta['gameId'],
                    order, name, pos
                ] + perf)

            except Exception as e:
                continue

    return all_rows

# Set meta manually or from JSON top-level keys
meta = {
    "leId": data.get("LE_ID", 1),
    "srId": data.get("SR_ID", 0),
    "seasonId": data.get("SEASON_ID", 2025),
    "gameId": data.get("G_ID", "unknown")
}

parsed = parse_arr_hitter(data, meta)
columns = [
    "leId", "srId", "seasonId", "gameId",
    "Order", "Name", "Position",
    "AB", "H", "RBI", "R", "AVG"
]
df = pd.DataFrame(parsed, columns=columns)

df.to_csv("combined_batters.csv", index=False, encoding="utf-8-sig")
print(" CSV saved.")


In [ ]:
import requests
import json
import pandas as pd
from datetime import date, timedelta
import time

url = "https://www.koreabaseball.com/ws/Main.asmx/GetKboGameList"
headers = {
    "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
    "X-Requested-With": "XMLHttpRequest",
    "User-Agent": "Mozilla/5.0"
}

start_date = date(2024, 3, 1)
end_date = date(2024, 11, 30)
delta = timedelta(days=1)

all_games = []

while start_date <= end_date:
    formatted_date = start_date.strftime("%Y%m%d")
    print(f" Fetching {formatted_date}")
    
    payload = {
        "leId": "1",
        "srId": "0",
        "seasonId": "2024",
        "date": formatted_date,
    }

    try:
        resp = requests.post(url, headers=headers, data=payload)
        resp.raise_for_status()
        data = resp.json()
        
        for game in data["game"]:
            all_games.append({
                "date": game["G_DT"],
                "gameId": game["G_ID"],
                "home_team": game["HOME_NM"],
                "away_team": game["AWAY_NM"],
                "stadium": game["S_NM"],
                "home_score": game["B_SCORE_CN"],
                "away_score": game["T_SCORE_CN"],
                "cancel_status": game["CANCEL_SC_NM"],
                "leId": game["LE_ID"],
                "srId": game["SR_ID"],
                "seasonId": game["SEASON_ID"]
            })

    except Exception as e:
        print(f"Error on {formatted_date}: {e}")
    
    time.sleep(0.5)
    start_date += delta

df = pd.DataFrame(all_games)
df.to_csv("kbo_2024_game_list.csv", index=False, encoding="utf-8-sig")
print("Saved to kbo_2024_game_list.csv")


In [ ]:
import requests
import pandas as pd
import json
import time
from tqdm import tqdm

# 讀入 gameId 清單
games_df = pd.read_csv("kbo_2024_game_list.csv")

# API 設定
url = "https://www.koreabaseball.com/ws/Schedule.asmx/GetBoxScoreScroll"
headers = {
    "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
    "X-Requested-With": "XMLHttpRequest",
    "User-Agent": "Mozilla/5.0"
}

# 儲存所有資料
all_hitter_data = []

for _, row in tqdm(games_df.iterrows(), total=len(games_df)):
    payload = {
        "leId": str(row["leId"]),
        "srId": str(row["srId"]),
        "seasonId": str(row["seasonId"]),
        "gameId": row["gameId"]
    }

    try:
        resp = requests.post(url, headers=headers, data=payload)
        resp.raise_for_status()
        content = resp.json()  #  now using the response directly

        for team_data in content.get("arrHitter", []):
            table1 = json.loads(team_data.get("table1", "{}"))
            table3 = json.loads(team_data.get("table3", "{}"))

            table1_rows = table1.get("rows", [])
            table3_rows = table3.get("rows", [])

            for i in range(min(len(table1_rows), len(table3_rows))):
                player_info = [cell["Text"].strip() for cell in table1_rows[i]["row"]]
                stats = [cell["Text"].strip() for cell in table3_rows[i]["row"]]
                record = {
                    "gameId": row["gameId"],
                    "date": row["date"],
                    "home_team": row["home_team"],
                    "away_team": row["away_team"],
                    "stadium": row["stadium"]
                }
                for idx, val in enumerate(player_info):
                    record[f"info_{idx+1}"] = val
                for idx, val in enumerate(stats):
                    record[f"stat_{idx+1}"] = val

                all_hitter_data.append(record)

    except Exception as e:
        print(f" Error on {row['gameId']}: {e}")
    
    time.sleep(0.5)

# 存成 CSV
pd.DataFrame(all_hitter_data).to_csv("kbo_2024_hitter_stats.csv", index=False, encoding="utf-8-sig")
print(" Done: kbo_2024_hitter_stats.csv")


In [ ]:
import requests
import json
import pandas as pd
from datetime import date, timedelta
import time

url = "https://www.koreabaseball.com/ws/Main.asmx/GetKboGameList"
headers = {
    "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
    "X-Requested-With": "XMLHttpRequest",
    "User-Agent": "Mozilla/5.0"
}

start_date = date(2023, 3, 1)
end_date = date(2023, 11, 30)
delta = timedelta(days=1)

all_games = []

while start_date <= end_date:
    formatted_date = start_date.strftime("%Y%m%d")
    print(f" Fetching {formatted_date}")
    
    payload = {
        "leId": "1",
        "srId": "0",
        "seasonId": "2023",
        "date": formatted_date,
    }

    try:
        resp = requests.post(url, headers=headers, data=payload)
        resp.raise_for_status()
        data = resp.json()
        
        for game in data["game"]:
            all_games.append({
                "date": game["G_DT"],
                "gameId": game["G_ID"],
                "home_team": game["HOME_NM"],
                "away_team": game["AWAY_NM"],
                "stadium": game["S_NM"],
                "home_score": game["B_SCORE_CN"],
                "away_score": game["T_SCORE_CN"],
                "cancel_status": game["CANCEL_SC_NM"],
                "leId": game["LE_ID"],
                "srId": game["SR_ID"],
                "seasonId": game["SEASON_ID"]
            })

    except Exception as e:
        print(f" Error on {formatted_date}: {e}")
    
    time.sleep(0.5)
    start_date += delta

df = pd.DataFrame(all_games)
df.to_csv("kbo_2023_game_list.csv", index=False, encoding="utf-8-sig")
print("Saved to kbo_2023_game_list.csv")

In [ ]:
import requests
import pandas as pd
import json
import time
from tqdm import tqdm

# 讀入 gameId 清單
games_df = pd.read_csv("kbo_2023_game_list.csv")

# API 設定
url = "https://www.koreabaseball.com/ws/Schedule.asmx/GetBoxScoreScroll"
headers = {
    "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
    "X-Requested-With": "XMLHttpRequest",
    "User-Agent": "Mozilla/5.0"
}

# 儲存所有資料
all_hitter_data = []

for _, row in tqdm(games_df.iterrows(), total=len(games_df)):
    payload = {
        "leId": str(row["leId"]),
        "srId": str(row["srId"]),
        "seasonId": str(row["seasonId"]),
        "gameId": row["gameId"]
    }

    try:
        resp = requests.post(url, headers=headers, data=payload)
        resp.raise_for_status()
        content = resp.json()  # now using the response directly

        for team_data in content.get("arrHitter", []):
            table1 = json.loads(team_data.get("table1", "{}"))
            table3 = json.loads(team_data.get("table3", "{}"))

            table1_rows = table1.get("rows", [])
            table3_rows = table3.get("rows", [])

            for i in range(min(len(table1_rows), len(table3_rows))):
                player_info = [cell["Text"].strip() for cell in table1_rows[i]["row"]]
                stats = [cell["Text"].strip() for cell in table3_rows[i]["row"]]
                record = {
                    "gameId": row["gameId"],
                    "date": row["date"],
                    "home_team": row["home_team"],
                    "away_team": row["away_team"],
                    "stadium": row["stadium"]
                }
                for idx, val in enumerate(player_info):
                    record[f"info_{idx+1}"] = val
                for idx, val in enumerate(stats):
                    record[f"stat_{idx+1}"] = val

                all_hitter_data.append(record)

    except Exception as e:
        print(f" Error on {row['gameId']}: {e}")
    
    time.sleep(0.5)

# 存成 CSV
pd.DataFrame(all_hitter_data).to_csv("kbo_2023_hitter_stats.csv", index=False, encoding="utf-8-sig")
print(" Done: kbo_2023_hitter_stats.csv")


In [ ]:
import requests
import json
import pandas as pd
from datetime import date, timedelta
import time

url = "https://www.koreabaseball.com/ws/Main.asmx/GetKboGameList"
headers = {
    "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
    "X-Requested-With": "XMLHttpRequest",
    "User-Agent": "Mozilla/5.0"
}

start_date = date(2022, 3, 1)
end_date = date(2022, 11, 30)
delta = timedelta(days=1)

all_games = []

while start_date <= end_date:
    formatted_date = start_date.strftime("%Y%m%d")
    print(f"📅 Fetching {formatted_date}")
    
    payload = {
        "leId": "1",
        "srId": "0",
        "seasonId": "2022",
        "date": formatted_date,
    }

    try:
        resp = requests.post(url, headers=headers, data=payload)
        resp.raise_for_status()
        data = resp.json()
        
        for game in data["game"]:
            all_games.append({
                "date": game["G_DT"],
                "gameId": game["G_ID"],
                "home_team": game["HOME_NM"],
                "away_team": game["AWAY_NM"],
                "stadium": game["S_NM"],
                "home_score": game["B_SCORE_CN"],
                "away_score": game["T_SCORE_CN"],
                "cancel_status": game["CANCEL_SC_NM"],
                "leId": game["LE_ID"],
                "srId": game["SR_ID"],
                "seasonId": game["SEASON_ID"]
            })

    except Exception as e:
        print(f" Error on {formatted_date}: {e}")
    
    time.sleep(0.5)
    start_date += delta

df = pd.DataFrame(all_games)
df.to_csv("kbo_2022_game_list.csv", index=False, encoding="utf-8-sig")
print("Saved to kbo_2022_game_list.csv")

In [ ]:
import requests
import pandas as pd
import json
import time
from tqdm import tqdm

# 讀入 gameId 清單
games_df = pd.read_csv("kbo_2022_game_list.csv")

# API 設定
url = "https://www.koreabaseball.com/ws/Schedule.asmx/GetBoxScoreScroll"
headers = {
    "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
    "X-Requested-With": "XMLHttpRequest",
    "User-Agent": "Mozilla/5.0"
}

# 儲存所有資料
all_hitter_data = []

for _, row in tqdm(games_df.iterrows(), total=len(games_df)):
    payload = {
        "leId": str(row["leId"]),
        "srId": str(row["srId"]),
        "seasonId": str(row["seasonId"]),
        "gameId": row["gameId"]
    }

    try:
        resp = requests.post(url, headers=headers, data=payload)
        resp.raise_for_status()
        content = resp.json()  #  now using the response directly

        for team_data in content.get("arrHitter", []):
            table1 = json.loads(team_data.get("table1", "{}"))
            table3 = json.loads(team_data.get("table3", "{}"))

            table1_rows = table1.get("rows", [])
            table3_rows = table3.get("rows", [])

            for i in range(min(len(table1_rows), len(table3_rows))):
                player_info = [cell["Text"].strip() for cell in table1_rows[i]["row"]]
                stats = [cell["Text"].strip() for cell in table3_rows[i]["row"]]
                record = {
                    "gameId": row["gameId"],
                    "date": row["date"],
                    "home_team": row["home_team"],
                    "away_team": row["away_team"],
                    "stadium": row["stadium"]
                }
                for idx, val in enumerate(player_info):
                    record[f"info_{idx+1}"] = val
                for idx, val in enumerate(stats):
                    record[f"stat_{idx+1}"] = val

                all_hitter_data.append(record)

    except Exception as e:
        print(f" Error on {row['gameId']}: {e}")
    
    time.sleep(0.5)

# 存成 CSV
pd.DataFrame(all_hitter_data).to_csv("kbo_2022_hitter_stats.csv", index=False, encoding="utf-8-sig")
print(" Done: kbo_2022_hitter_stats.csv")


## append crowd amount

In [ ]:
import requests
import pandas as pd
import time
from tqdm import tqdm

# Load game list
games_df = pd.read_csv("kbo_2024_game_list.csv")

# API info
url = "https://www.koreabaseball.com/ws/Schedule.asmx/GetScoreBoardScroll"
headers = {
    "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
    "X-Requested-With": "XMLHttpRequest",
    "User-Agent": "Mozilla/5.0"
}

# Store results
crowd_data = []

for _, row in tqdm(games_df.iterrows(), total=len(games_df)):
    payload = {
        "leId": str(row["leId"]),
        "srId": str(row["srId"]),
        "seasonId": str(row["seasonId"]),
        "gameId": row["gameId"]
    }

    try:
        resp = requests.post(url, headers=headers, data=payload)
        resp.raise_for_status()
        data = resp.json()

        # Extract relevant info
        record = {
            "gameId": row["gameId"],
            "date": row["date"],
            "home_team": data.get("HOME_NM", ""),
            "away_team": data.get("AWAY_NM", ""),
            "stadium": data.get("S_NM", ""),
            "crowd": data.get("CROWD_CN", ""),
            "home_score": data.get("B_SCORE_CN", ""),
            "away_score": data.get("T_SCORE_CN", ""),
            "start_time": data.get("START_TM", ""),
            "end_time": data.get("END_TM", ""),
            "duration": data.get("USE_TM", "")
        }

        crowd_data.append(record)

    except Exception as e:
        print(f" Error on {row['gameId']}: {e}")
    
    time.sleep(0.3)  # Gentle on the server

# Save as CSV
pd.DataFrame(crowd_data).to_csv("kbo_2024_game_crowd_stats.csv", index=False, encoding="utf-8-sig")
print(" Done: kbo_2024_game_crowd_stats.csv")


In [ ]:
import requests
import pandas as pd
import time
from tqdm import tqdm

# Load game list
games_df = pd.read_csv("kbo_2023_game_list.csv")

# API info
url = "https://www.koreabaseball.com/ws/Schedule.asmx/GetScoreBoardScroll"
headers = {
    "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
    "X-Requested-With": "XMLHttpRequest",
    "User-Agent": "Mozilla/5.0"
}

# Store results
crowd_data = []

for _, row in tqdm(games_df.iterrows(), total=len(games_df)):
    payload = {
        "leId": str(row["leId"]),
        "srId": str(row["srId"]),
        "seasonId": str(row["seasonId"]),
        "gameId": row["gameId"]
    }

    try:
        resp = requests.post(url, headers=headers, data=payload)
        resp.raise_for_status()
        data = resp.json()

        # Extract relevant info
        record = {
            "gameId": row["gameId"],
            "date": row["date"],
            "home_team": data.get("HOME_NM", ""),
            "away_team": data.get("AWAY_NM", ""),
            "stadium": data.get("S_NM", ""),
            "crowd": data.get("CROWD_CN", ""),
            "home_score": data.get("B_SCORE_CN", ""),
            "away_score": data.get("T_SCORE_CN", ""),
            "start_time": data.get("START_TM", ""),
            "end_time": data.get("END_TM", ""),
            "duration": data.get("USE_TM", "")
        }

        crowd_data.append(record)

    except Exception as e:
        print(f" Error on {row['gameId']}: {e}")
    
    time.sleep(0.3)  # Gentle on the server

# Save as CSV
pd.DataFrame(crowd_data).to_csv("kbo_2023_game_crowd_stats.csv", index=False, encoding="utf-8-sig")
print("Done: kbo_2023_game_crowd_stats.csv")


In [ ]:
import requests
import pandas as pd
import time
from tqdm import tqdm

# Load game list
games_df = pd.read_csv("kbo_2022_game_list.csv")

# API info
url = "https://www.koreabaseball.com/ws/Schedule.asmx/GetScoreBoardScroll"
headers = {
    "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
    "X-Requested-With": "XMLHttpRequest",
    "User-Agent": "Mozilla/5.0"
}

# Store results
crowd_data = []

for _, row in tqdm(games_df.iterrows(), total=len(games_df)):
    payload = {
        "leId": str(row["leId"]),
        "srId": str(row["srId"]),
        "seasonId": str(row["seasonId"]),
        "gameId": row["gameId"]
    }

    try:
        resp = requests.post(url, headers=headers, data=payload)
        resp.raise_for_status()
        data = resp.json()

        # Extract relevant info
        record = {
            "gameId": row["gameId"],
            "date": row["date"],
            "home_team": data.get("HOME_NM", ""),
            "away_team": data.get("AWAY_NM", ""),
            "stadium": data.get("S_NM", ""),
            "crowd": data.get("CROWD_CN", ""),
            "home_score": data.get("B_SCORE_CN", ""),
            "away_score": data.get("T_SCORE_CN", ""),
            "start_time": data.get("START_TM", ""),
            "end_time": data.get("END_TM", ""),
            "duration": data.get("USE_TM", "")
        }

        crowd_data.append(record)

    except Exception as e:
        print(f"Error on {row['gameId']}: {e}")
    
    time.sleep(0.3)  # Gentle on the server

# Save as CSV
pd.DataFrame(crowd_data).to_csv("kbo_2022_game_crowd_stats.csv", index=False, encoding="utf-8-sig")
print(" Done: kbo_2022_game_crowd_stats.csv")


## change the column name

In [ ]:
import pandas as pd

df = pd.read_csv("kbo_2024_hitter_stats.csv")

# Rename the columns
df = df.rename(columns={
    "info_1": "타순",
    "info_2": "수비위치",
    "info_3": "선수명",
    "stat_1": "타수",
    "stat_2": "안타",
    "stat_3": "타점",
    "stat_4": "득점",
    "stat_5": "타율"
})

# Save updated CSV
df.to_csv("kbo_2024_hitter_stats_renamed.csv", index=False, encoding="utf-8-sig")

In [ ]:
import pandas as pd

df = pd.read_csv("kbo_2023_hitter_stats.csv")

# Rename the columns
df = df.rename(columns={
    "info_1": "타순",
    "info_2": "수비위치",
    "info_3": "선수명",
    "stat_1": "타수",
    "stat_2": "안타",
    "stat_3": "타점",
    "stat_4": "득점",
    "stat_5": "타율"
})

# Save updated CSV
df.to_csv("kbo_2023_hitter_stats_renamed.csv", index=False, encoding="utf-8-sig")


In [ ]:
import pandas as pd

df = pd.read_csv("kbo_2022_hitter_stats.csv")

# Rename the columns
df = df.rename(columns={
    "info_1": "타순",
    "info_2": "수비위치",
    "info_3": "선수명",
    "stat_1": "타수",
    "stat_2": "안타",
    "stat_3": "타점",
    "stat_4": "득점",
    "stat_5": "타율"
})

# Save updated CSV
df.to_csv("kbo_2022_hitter_stats_renamed.csv", index=False, encoding="utf-8-sig")


## add team_name column

In [ ]:
import pandas as pd

df = pd.read_csv("kbo_2024_hitter_stats_renamed.csv")

# Add a column for team_name
df["team_name"] = ""

# Define function to split based on 타순 transition 9 → 1
def assign_team_by_batting_order(group):
    group = group.reset_index(drop=True)
    boundary_idx = None

    # Look for the transition from 타순 9 → 1
    for i in range(len(group) - 1):
        if group.loc[i, "타순"] == 9 and group.loc[i + 1, "타순"] == 1:
            boundary_idx = i
            break

    if boundary_idx is not None:
        group.loc[:boundary_idx, "team_name"] = group.loc[0, "away_team"]
        group.loc[boundary_idx + 1:, "team_name"] = group.loc[0, "home_team"]
    else:
        print(f" No 9→1 transition found in gameId {group.loc[0, 'gameId']}")
        # Default fallback (equal split)
        half = len(group) // 2
        group.loc[:half, "team_name"] = group.loc[0, "away_team"]
        group.loc[half + 1:, "team_name"] = group.loc[0, "home_team"]

    return group

# Apply to each gameId group
df = df.groupby("gameId", group_keys=False).apply(assign_team_by_batting_order)

# Optional: add team_type column
df["team_type"] = df.apply(lambda row: "away" if row["team_name"] == row["away_team"] else "home", axis=1)

# Save the result
df.to_csv("kbo_2024_hitter_stats_team_by_batting_order.csv", index=False, encoding="utf-8-sig")
print("Accurate team_name assignment based on 타순 transition.")


In [ ]:
import pandas as pd

df = pd.read_csv("kbo_2023_hitter_stats_renamed.csv")

# Add a column for team_name
df["team_name"] = ""

# Define function to split based on 타순 transition 9 → 1
def assign_team_by_batting_order(group):
    group = group.reset_index(drop=True)
    boundary_idx = None

    # Look for the transition from 타순 9 → 1
    for i in range(len(group) - 1):
        if group.loc[i, "타순"] == 9 and group.loc[i + 1, "타순"] == 1:
            boundary_idx = i
            break

    if boundary_idx is not None:
        group.loc[:boundary_idx, "team_name"] = group.loc[0, "away_team"]
        group.loc[boundary_idx + 1:, "team_name"] = group.loc[0, "home_team"]
    else:
        print(f" No 9→1 transition found in gameId {group.loc[0, 'gameId']}")
        # Default fallback (equal split)
        half = len(group) // 2
        group.loc[:half, "team_name"] = group.loc[0, "away_team"]
        group.loc[half + 1:, "team_name"] = group.loc[0, "home_team"]

    return group

# Apply to each gameId group
df = df.groupby("gameId", group_keys=False).apply(assign_team_by_batting_order)

# Optional: add team_type column
df["team_type"] = df.apply(lambda row: "away" if row["team_name"] == row["away_team"] else "home", axis=1)

# Save the result
df.to_csv("kbo_2023_hitter_stats_team_by_batting_order.csv", index=False, encoding="utf-8-sig")
print(" Accurate team_name assignment based on 타순 transition.")


In [ ]:
import pandas as pd

df = pd.read_csv("kbo_2022_hitter_stats_renamed.csv")

# Add a column for team_name
df["team_name"] = ""

# Define function to split based on 타순 transition 9 → 1
def assign_team_by_batting_order(group):
    group = group.reset_index(drop=True)
    boundary_idx = None

    # Look for the transition from 타순 9 → 1
    for i in range(len(group) - 1):
        if group.loc[i, "타순"] == 9 and group.loc[i + 1, "타순"] == 1:
            boundary_idx = i
            break

    if boundary_idx is not None:
        group.loc[:boundary_idx, "team_name"] = group.loc[0, "away_team"]
        group.loc[boundary_idx + 1:, "team_name"] = group.loc[0, "home_team"]
    else:
        print(f" No 9→1 transition found in gameId {group.loc[0, 'gameId']}")
        # Default fallback (equal split)
        half = len(group) // 2
        group.loc[:half, "team_name"] = group.loc[0, "away_team"]
        group.loc[half + 1:, "team_name"] = group.loc[0, "home_team"]

    return group

# Apply to each gameId group
df = df.groupby("gameId", group_keys=False).apply(assign_team_by_batting_order)

# Optional: add team_type column
df["team_type"] = df.apply(lambda row: "away" if row["team_name"] == row["away_team"] else "home", axis=1)

# Save the result
df.to_csv("kbo_2022_hitter_stats_team_by_batting_order.csv", index=False, encoding="utf-8-sig")
print("Accurate team_name assignment based on 타순 transition.")


## list all unique hitters by team

In [ ]:
import pandas as pd

# Load your CSV file
df = pd.read_csv("kbo_2024_hitter_stats_team_by_batting_order.csv")

# Group by team_name and get unique hitters
team_hitters_overall = df.groupby("team_name")["선수명"].unique().reset_index()

# # Optional: sort hitters alphabetically for each team
# team_hitters_overall["선수명"] = team_hitters_overall["선수명"].apply(lambda x: sorted(x))

# Rename for clarity
team_hitters_overall = team_hitters_overall.rename(columns={"선수명": "hitters"})

# Display
print(team_hitters_overall)

# Optional: Save to CSV
team_hitters_overall.to_csv("2024_team_hitters_overall.csv", index=False, encoding="utf-8-sig")


In [ ]:
import pandas as pd

# Load your CSV file
df = pd.read_csv("kbo_2023_hitter_stats_team_by_batting_order.csv")

# Group by team_name and get unique hitters
team_hitters_overall = df.groupby("team_name")["선수명"].unique().reset_index()

# # Optional: sort hitters alphabetically for each team
# team_hitters_overall["선수명"] = team_hitters_overall["선수명"].apply(lambda x: sorted(x))

# Rename for clarity
team_hitters_overall = team_hitters_overall.rename(columns={"선수명": "hitters"})

# Display
print(team_hitters_overall)

# Optional: Save to CSV
team_hitters_overall.to_csv("2023_team_hitters_overall.csv", index=False, encoding="utf-8-sig")


In [ ]:
import pandas as pd

# Load your CSV file
df = pd.read_csv("kbo_2022_hitter_stats_team_by_batting_order.csv")

# Group by team_name and get unique hitters
team_hitters_overall = df.groupby("team_name")["선수명"].unique().reset_index()

# # Optional: sort hitters alphabetically for each team
# team_hitters_overall["선수명"] = team_hitters_overall["선수명"].apply(lambda x: sorted(x))

# Rename for clarity
team_hitters_overall = team_hitters_overall.rename(columns={"선수명": "hitters"})

# Display
print(team_hitters_overall)

# Optional: Save to CSV
team_hitters_overall.to_csv("2022_team_hitters_overall.csv", index=False, encoding="utf-8-sig")


In [ ]:
import pandas as pd

# Load each year's file
df_2022 = pd.read_csv("2022_team_hitters_overall.csv")
df_2023 = pd.read_csv("2023_team_hitters_overall.csv")
df_2024 = pd.read_csv("2024_team_hitters_overall.csv")

# Rename hitter column to year
df_2022 = df_2022.rename(columns={"hitters": "2022"})
df_2023 = df_2023.rename(columns={"hitters": "2023"})
df_2024 = df_2024.rename(columns={"hitters": "2024"})

# Merge on team_name
merged = pd.merge(df_2022, df_2023, on="team_name", how="outer")
merged = pd.merge(merged, df_2024, on="team_name", how="outer")

# Optional: sort by team name
merged = merged.sort_values("team_name").reset_index(drop=True)

# Save to CSV
merged.to_csv("team_hitters_2022_2024.csv", index=False, encoding="utf-8-sig")

# Display a sample
print(merged.head())


## Merge crowd and hitter csv file

In [ ]:
import pandas as pd

# Load only the required data
crowd_2022 = pd.read_csv("kbo_2022_game_crowd_stats.csv")
crowd_2023 = pd.read_csv("kbo_2023_game_crowd_stats.csv")
crowd_2024 = pd.read_csv("kbo_2024_game_crowd_stats.csv")

hitter_2022 = pd.read_csv("kbo_2022_hitter_stats_team_by_batting_order.csv")
hitter_2023 = pd.read_csv("kbo_2023_hitter_stats_team_by_batting_order.csv")
hitter_2024 = pd.read_csv("kbo_2024_hitter_stats_team_by_batting_order.csv")

# Combine crowd stats and keep only selected columns
crowd_all = pd.concat([crowd_2022, crowd_2023, crowd_2024], ignore_index=True)
crowd_selected = crowd_all[['gameId', 'stadium', 'crowd', 'duration', 'start_time', 'end_time']]

# Combine hitter stats
hitter_all = pd.concat([hitter_2022, hitter_2023, hitter_2024], ignore_index=True)

# Merge on gameId
merged_df = pd.merge(hitter_all, crowd_selected, on='gameId', how='left')

# Save result
merged_df.to_csv("merged_kbo_game_data_2022_2024.csv", index=False)


In [ ]:
import pandas as pd

# Load CSV
df = pd.read_csv("KBO_타자_등장곡_응원가 - 工作表1-2.csv")

# Only split when ' - ' exists
def split_song_title(value):
    if isinstance(value, str) and ' - ' in value:
        return pd.Series(value.split(' - ', 1))
    else:
        return pd.Series([None, value])

df[['singer', 'original_song_title']] = df['original_song_title'].apply(split_song_title)

# Flags from Note column
df['reuse_flag'] = df['Note'].apply(lambda x: 1 if isinstance(x, str) and '재사용' in x else 0)
df['inherit_flag'] = df['Note'].apply(lambda x: 1 if isinstance(x, str) and '물려' in x else 0)

# Reorder columns:
cols = df.columns.tolist()

# Ensure no duplication
cols = [col for col in cols if col not in ['singer', 'reuse_flag', 'inherit_flag']]

# Insert 'singer' next to 'original_song_title'
title_index = cols.index('original_song_title')
cols.insert(title_index, 'singer')

# Insert flags before 'Note'
note_index = cols.index('Note')
cols.insert(note_index, 'inherit_flag')
cols.insert(note_index, 'reuse_flag')

# Reorder dataframe
df = df[cols]

# Save
df.to_csv("processed_kbo_song_data.csv", index=False)

In [ ]:
import pandas as pd

# Load your datasets
song_df = pd.read_csv("KBO_fight song_lyrics - processed_kbo_song_data.csv")
performance_df = pd.read_csv("merged_kbo_game_data_2022_2024.csv")

# Prepare the song dataset with only necessary columns
# merge using: Player_name → 선수명, Team → team_name
song_df_subset = song_df[[
    'Player_name', 'Team', 'original_song_title', 'original_song_singer',
    'reuse_flag', 'inherit_flag', 'shared_player_count', 'Song_lyrics'
]]

# Merge: match song_df.Player_name with performance_df.선수명
# and song_df.Team with performance_df.team_name
merged = pd.merge(
    performance_df,
    song_df_subset,
    left_on=['선수명', 'team_name'],
    right_on=['Player_name', 'Team'],
    how='left'
)

# Optional: drop duplicate key columns after merge
merged.drop(['Player_name', 'Team'], axis=1, inplace=True)

# Save the result
merged.to_csv("merged_kbo_game_data_with_song_info.csv", index=False)


In [ ]:
import pandas as pd

# Load the merged data
df = pd.read_csv("merged_kbo_game_data_with_song_info.csv")

# 1. Replace NaN in shared_song_count with 0
df['shared_player_count'] = df['shared_player_count'].fillna(0).astype(int)

# 2. Replace NaN in Song_lyrics with "none"
df['Song_lyrics'] = df['Song_lyrics'].fillna("none")

# Save the cleaned result
df.to_csv("merged_kbo_game_data_with_song_info_final.csv", index=False)


## Sentiment analysis

count the freq of lyrics

In [ ]:
# 1. Install required packages (if you haven't already):
#    pip install kiwipiepy pandas

from kiwipiepy import Kiwi
import pandas as pd
from collections import Counter

# 2. Load your merged dataset
df = pd.read_csv('merged_kbo_game_data_with_song_info_final.csv')

# 3. Initialize Kiwi tokenizer
kiwi = Kiwi()

# 4. Define a tokenization function
def tokenize(text: str) -> list:
    if not isinstance(text, str) or not text.strip():
        return []
    return [token[0] for token in kiwi.tokenize(text)]

# 5. Tokenize all lyrics and count frequencies
all_tokens = df['Song_lyrics'].apply(tokenize).explode().dropna()
token_freq = Counter(all_tokens)

# 6. (Optional) Convert to DataFrame for easier viewing/saving
freq_df = pd.DataFrame(token_freq.most_common(), columns=['token', 'frequency'])

# 7. View the top N tokens
print(freq_df.head(50))

# 8. Save to CSV if you like
freq_df.to_csv('song_lyrics_token_frequencies.csv', index=False)


count the sentiment score

In [ ]:
import pandas as pd
import re
from collections import defaultdict, ChainMap
from kiwipiepy import Kiwi

# 1. Load merged song lyric dataset
df = pd.read_csv("merged_kbo_game_data_with_song_info_final.csv")

# 2. Download & load NSMC
nsmc = pd.read_csv("https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt", sep="\t").dropna()

# 3. Extract NSMC-based sentiment dictionary
kiwi = Kiwi()
pos_words = defaultdict(int)
neg_words = defaultdict(int)

for _, row in nsmc.iterrows():
    label = row['label']
    text = row['document']
    for tok in kiwi.tokenize(text):
        if tok.tag.startswith('N') or tok.tag.startswith('V'):
            if label == 1:
                pos_words[tok.form] += 1
            else:
                neg_words[tok.form] += 1

nsmc_sentiment_dict = {}
for word in set(pos_words) | set(neg_words):
    score = pos_words[word] - neg_words[word]
    if abs(score) >= 3:  # optional: threshold to keep strong polarities
        nsmc_sentiment_dict[word] = 1 if score > 0 else -1

# 4. Custom lyric-specific sentiment dictionary
custom_sentiment_dict = {
    '가자': 1, '파이팅': 1, '승리': 2, '우승': 2, '홈런': 2,
    '힘내': 1, '날리': 2, '달리': 2, '시원하': 1, '최강': 2,
    '워': 1, '어': 1, '오': 1, '아': 1,  # basic boost roots
    '안타': 2, '어라': 1, '버리': 2, '힘차': 1, '외치': 1,
    '쌔리라': 1, '날아오르': 2,
    '삼성': 1, '롯데': 1, 'LG': 1, '한화': 1, '두산': 1, 'NC': 1,
    '히어로즈': 1, '이글스': 1, '자이언츠': 1, '무적': 1, '타이거즈': 1, '다이노스': 1,
}

# 5. Combine dictionaries
sentiment_dict = defaultdict(int, ChainMap(custom_sentiment_dict, nsmc_sentiment_dict))

# 6. Boost patterns
boost_patterns = [
    re.compile(r'가자[아어오]*'),        # 가자아아, 가자어어
    re.compile(r'워[어오아으아]{1,}'),    # 워어어, 워오오
    re.compile(r'아{2,}'),              # 아아아
    re.compile(r'어{2,}'),              # 어어어
    re.compile(r'오{2,}'),              # 오오오
    re.compile(r'G[oO]+'),             # Gooo, GOOOOO
    re.compile(r'워[우어오]{1,}워'),     # 워우워, 워어워
    re.compile(r'[가-힣]+라$')           # 올라라, 뛰어라 → 激勵語尾
]

def is_boost_marker(text):
    return any(p.fullmatch(text) for p in boost_patterns)

# 7. Tokenization with punctuation and brackets
token_pattern = re.compile(r'\([^)]*\)|[！～]|\w+')

def tokenize_with_meta(text):
    if not isinstance(text, str) or not text.strip():
        return [], []
    tokens = []
    for m in token_pattern.finditer(text):
        tokens.append({'text': m.group(), 'start': m.start(), 'end': m.end()})
    paren_spans = [(t['start'], t['end']) for t in tokens if t['text'].startswith('(')]
    return tokens, paren_spans

# 8. Sentiment score computation
def compute_sentiment(text):
    tokens, paren_spans = tokenize_with_meta(text)
    if not tokens:
        return 0.0
    scores = []
    for i, token in enumerate(tokens):
        tok = token['text']
        s, e = token['start'], token['end']

        if is_boost_marker(tok) or tok in {'！', '～'}:
            continue

        kiwi_tokens = kiwi.tokenize(tok)
        base = sum([sentiment_dict[kw.form] for kw in kiwi_tokens if sentiment_dict[kw.form] != 0])
        if base == 0:
            continue

        boost = False
        if (s > 0 and text[s - 1] in '！～') or (e < len(text) and text[e] in '！～'):
            boost = True
        if any(s >= ps and e <= pe for ps, pe in paren_spans):
            boost = True
        for j in range(max(0, i - 2), i):
            if is_boost_marker(tokens[j]['text']):
                boost = True

        score = base * (2 if boost else 1)
        scores.append(score)
    return sum(scores) / len(scores) if scores else 0.0

# 9. Apply to song lyrics
df['kiwi_sentiment_boost'] = df['Song_lyrics'].apply(compute_sentiment)

# 10. Save result
df.to_csv("with_kiwi_sentiment_boost_final.csv", index=False)


only use self-defined word

In [ ]:
import pandas as pd
import re
from collections import defaultdict
from kiwipiepy import Kiwi

# 1. Load your merged dataset
df = pd.read_csv("merged_kbo_game_data_with_song_info_final.csv")

# 2. Initialize kiwi
kiwi = Kiwi()

# 3. Define custom sentiment dictionary (no NSMC)
custom_sentiment_dict = {
    '가자': 1, '파이팅': 1, '승리': 2, '우승': 2, '홈런': 2,
    '힘내': 1, '날리': 2, '달리': 2, '시원하': 1, '최강': 2,
    '워': 1, '어': 1, '오': 1, '아': 1,
    '안타': 2, '어라': 1, '버리': 2, '힘차': 1, '외치': 1,
    '쌔리라': 1, '날아오르': 2,
    '삼성': 1, '롯데': 1, 'LG': 1, '한화': 1, '두산': 1, 'NC': 1,
    '히어로즈': 1, '이글스': 1, '자이언츠': 1, '무적': 1, '타이거즈': 1, '다이노스': 1,
}

# 4. Boost patterns
boost_patterns = [
    re.compile(r'가자[아어오]*'),
    re.compile(r'워[어오아으아]{1,}'),
    re.compile(r'아{2,}'),
    re.compile(r'어{2,}'),
    re.compile(r'오{2,}'),
    re.compile(r'G[oO]+'),
    re.compile(r'워[우어오]{1,}워'),
    re.compile(r'[가-힣]+라$')
]

def is_boost_marker(text):
    return any(p.fullmatch(text) for p in boost_patterns)

# 5. Tokenize with punctuation/boost marker brackets
token_pattern = re.compile(r'\([^)]*\)|[！～]|\w+')

def tokenize_with_meta(text):
    if not isinstance(text, str) or not text.strip():
        return [], []
    tokens = []
    for m in token_pattern.finditer(text):
        tokens.append({'text': m.group(), 'start': m.start(), 'end': m.end()})
    paren_spans = [(t['start'], t['end']) for t in tokens if t['text'].startswith('(')]
    return tokens, paren_spans

# 6. Compute sentiment (only using custom dict)
def compute_custom_sentiment(text):
    tokens, paren_spans = tokenize_with_meta(text)
    if not tokens:
        return 0.0
    scores = []
    for i, token in enumerate(tokens):
        tok = token['text']
        s, e = token['start'], token['end']

        if is_boost_marker(tok) or tok in {'！', '～'}:
            continue

        kiwi_tokens = kiwi.tokenize(tok)
        base = sum([custom_sentiment_dict.get(kw.form, 0) for kw in kiwi_tokens])
        if base == 0:
            continue

        boost = False
        if (s > 0 and text[s - 1] in '！～') or (e < len(text) and text[e] in '！～'):
            boost = True
        if any(s >= ps and e <= pe for ps, pe in paren_spans):
            boost = True
        for j in range(max(0, i - 2), i):
            if is_boost_marker(tokens[j]['text']):
                boost = True

        score = base * (2 if boost else 1)
        scores.append(score)
    return sum(scores) / len(scores) if scores else 0.0

# 7. Apply to your song lyric column
df['custom_sentiment_score'] = df['Song_lyrics'].apply(compute_custom_sentiment)

# 8. Save result
df.to_csv("with_custom_sentiment_only.csv", index=False)


In [ ]:
import pandas as pd

# 1. 載入你已經完成情緒分析的 CSV（含 kiwi_sentiment_boost 欄位）
df = pd.read_csv("with_custom_sentiment_only.csv")

# 2. 四捨五入 kiwi_sentiment_boost 到小數點後兩位（float）
df['custom_sentiment_score'] = df['custom_sentiment_score'].round(2)

# 3. 儲存成新檔案
df.to_csv("with_sentiment_boost_rounded.csv", index=False)

# 4. 確認前幾筆資料（可選）
print(df[['선수명', 'gameId', 'custom_sentiment_score']].head())

## sentiment score for top/bottom 10 songs

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# Set font for Korean (macOS or Windows)
plt.rcParams['font.family'] = 'AppleGothic'  # macOS
# plt.rcParams['font.family'] = 'Malgun Gothic'  # ← if you're on Windows

plt.rcParams['axes.unicode_minus'] = False  # Ensure minus sign is displayed correctly


# 1. Load the dataset
df = pd.read_csv("with_sentiment_boost_rounded.csv")

# 2. Filter out rows without sentiment score
df = df[df['custom_sentiment_score'].notna()]

# 3. Select Top 10 players by highest score (no duplicates)
top10 = df.sort_values(by='custom_sentiment_score', ascending=False) \
          .drop_duplicates(subset='선수명') \
          .head(10)

# 4. Select Bottom 10 players by lowest *non-zero* score (no duplicates)
bottom10 = df[df['custom_sentiment_score'] != 0] \
             .sort_values(by='custom_sentiment_score', ascending=True) \
             .drop_duplicates(subset='선수명') \
             .head(10)

# 5. Add labels for grouping
top10['sentiment_group'] = 'Top 10'
bottom10['sentiment_group'] = 'Bottom 10'
combined = pd.concat([top10, bottom10], ignore_index=True)

# 6. Plot horizontal bar chart
plt.figure(figsize=(12, 6))
sns.barplot(data=combined,
            x='custom_sentiment_score',
            y='선수명',
            hue='sentiment_group',
            dodge=False,
            palette='coolwarm')
plt.title("Custom Sentiment Score: Top 10 vs Bottom 10 Players (Score ≠ 0)")
plt.xlabel("Custom Sentiment Score")
plt.ylabel("Player")
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

# 7. Print short lyrics preview for interpretation
for group_name, group_df in combined.groupby('sentiment_group'):
    print(f"\n====== {group_name} Lyrics Highlights ======\n")
    for idx, row in group_df.iterrows():
        print(f"[{row['선수명']}] {row['original_song_title']}")
        print(row['Song_lyrics'][:100].replace('\n', ' ') + "...\n")


wordclouds for top10/bottom10

In [ ]:
import pandas as pd
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from kiwipiepy import Kiwi

# 1. Load the dataset
df = pd.read_csv("with_sentiment_boost_rounded.csv")

# 2. Filter rows with valid sentiment score and lyrics
df = df[df['custom_sentiment_score'].notna() & df['Song_lyrics'].notna()]

# 3. Remove duplicate players and select top/bottom 10
top10 = df.sort_values(by='custom_sentiment_score', ascending=False) \
          .drop_duplicates(subset='선수명') \
          .head(10)

bottom10 = df[df['custom_sentiment_score'] != 0] \
             .sort_values(by='custom_sentiment_score', ascending=True) \
             .drop_duplicates(subset='선수명') \
             .head(10)

# 4. Extract lyrics
top_lyrics = top10['Song_lyrics'].astype(str)
bottom_lyrics = bottom10['Song_lyrics'].astype(str)

# 5. Tokenize lyrics using kiwi
kiwi = Kiwi()

def tokenize_lyrics(lyrics_series):
    tokens = []
    for line in lyrics_series:
        for token in kiwi.tokenize(line):
            if token.tag.startswith(('NN', 'VV', 'VA', 'MAG', 'IC')):  # keep only meaningful POS
                tokens.append(token.form)
    return " ".join(tokens)

top_text = tokenize_lyrics(top_lyrics)
bottom_text = tokenize_lyrics(bottom_lyrics)

# 6. Function to show wordcloud
def show_wordcloud(text, title):
    wc = WordCloud(
        font_path="/Library/Fonts/AppleGothic.ttf",  # For macOS. Use "C:/Windows/Fonts/malgun.ttf" on Windows
        background_color="white",
        width=800,
        height=400
    ).generate(text)
    plt.figure(figsize=(10, 5))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(title)
    plt.show()

# 7. Display both wordclouds
show_wordcloud(top_text, "Top 10 Songs (Custom Sentiment)")
show_wordcloud(bottom_text, "Bottom 10 Songs (Custom Sentiment, Excluding Score = 0)")


## EDA for each team

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load your dataset
df = pd.read_csv("with_sentiment_boost_rounded.csv")

# 2. Filter valid entries (score + lyrics not null)
df = df[df['custom_sentiment_score'].notna() & df['Song_lyrics'].notna()]

# 3. Remove duplicate players per team
df_unique = df.drop_duplicates(subset=['team_name', '선수명'])

# 4. Compute per-team average sentiment (unique players only)
team_avg = df_unique.groupby('team_name')['custom_sentiment_score'].mean().reset_index()

# 5. Plot: average sentiment per team
plt.figure(figsize=(12, 6))
sns.barplot(data=team_avg, x='team_name', y='custom_sentiment_score', palette='viridis')
plt.title("Average Custom Sentiment Score by Team (Unique Players)")
plt.xlabel("Team")
plt.ylabel("Average Sentiment Score")
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

# 6. Get Top 5 and Bottom 5 sentiment score players per team (include score=0)
top_bottom_df = []

for team in df_unique['team_name'].unique():
    team_df = df_unique[df_unique['team_name'] == team]
    
    top5 = team_df.sort_values(by='custom_sentiment_score', ascending=False).head(5)
    bottom5 = team_df.sort_values(by='custom_sentiment_score', ascending=True).head(5)
    
    top5['rank_type'] = 'Top 5'
    bottom5['rank_type'] = 'Bottom 5'
    
    top_bottom_df.append(pd.concat([top5, bottom5]))

ranked_df = pd.concat(top_bottom_df)

# 7. Plot: top 5 & bottom 5 sentiment players across all teams
plt.figure(figsize=(14, 8))
sns.barplot(data=ranked_df,
            x='custom_sentiment_score',
            y='선수명',
            hue='rank_type',
            palette={'Top 5': 'dodgerblue', 'Bottom 5': 'salmon'})
plt.title("Top 5 and Bottom 5 Player Sentiment Scores by Team (Score=0 Included)")
plt.xlabel("Custom Sentiment Score")
plt.ylabel("Player")
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Use the `ranked_df` you already created from the previous code

# Set up FacetGrid: one subplot per team
g = sns.FacetGrid(ranked_df, col='team_name', col_wrap=4, height=4, sharex=False, sharey=False)

# Draw horizontal barplot per team
g.map_dataframe(sns.barplot,
                x='custom_sentiment_score',
                y='선수명',
                hue='rank_type',
                dodge=False,
                palette={'Top 5': 'dodgerblue', 'Bottom 5': 'salmon'})

g.add_legend()
g.set_titles(col_template="{col_name}")
g.set_axis_labels("Sentiment Score", "Player")
for ax in g.axes.flat:
    ax.tick_params(axis='y', labelsize=8)
    ax.grid(axis='x', linestyle='--', alpha=0.4)

plt.subplots_adjust(top=0.9)
g.fig.suptitle("Top & Bottom 5 Sentiment Scores Per Team (Score=0 Included)", fontsize=16)
plt.show()


## descriptive statistics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore

# Load data
df = pd.read_csv("with_sentiment_boost_rounded.csv")
df = df[df['custom_sentiment_score'].notna() & df['Song_lyrics'].notna()]

# Descriptive statistics and histogram
print("Descriptive Stats for custom_sentiment_score:\n")
print(df['custom_sentiment_score'].describe())

# Create figure with two subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Histogram on first subplot
ax1.hist(df['custom_sentiment_score'], bins=30, color='skyblue', edgecolor='black')
ax1.set_title("Distribution of Custom Sentiment Scores")
ax1.set_xlabel("Sentiment Score")
ax1.set_ylabel("Frequency")
ax1.grid(True)

# Boxplot on second subplot
sns.boxplot(y=df['custom_sentiment_score'], ax=ax2, color='lightgreen')
ax2.set_title("Boxplot of Custom Sentiment Scores")
ax2.set_ylabel("Sentiment Score")
ax2.grid(True, axis='y')

plt.tight_layout()
plt.show()

# Additional boxplot by team
plt.figure(figsize=(14, 6))
sns.boxplot(data=df, x='team_name', y='custom_sentiment_score')
plt.title("Sentiment Score Distribution by Team")
plt.xlabel("Team")
plt.ylabel("Sentiment Score")
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()



In [ ]:
# Scatter plots with hitter performance
plt.figure(figsize=(6, 4))
sns.scatterplot(data=df, x='custom_sentiment_score', y='안타')
plt.title("Sentiment Score vs Hits (안타)")
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
sns.scatterplot(data=df, x='custom_sentiment_score', y='타율')
plt.title("Sentiment Score vs Batting Average (타율)")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Recompute team_type based on whether the player's team is home or away
def define_team_type(row):
    if row['team_name'] == row['home_team']:
        return 'home'
    elif row['team_name'] == row['away_team']:
        return 'away'
    else:
        return 'unknown'  # fallback for any data mismatch

df['team_type'] = df.apply(define_team_type, axis=1)
# Boxplot for sentiment by team type
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='team_type', y='custom_sentiment_score')
plt.title("Sentiment Score by Team Type (Home vs Away)")
plt.xlabel("Team Type")
plt.ylabel("Custom Sentiment Score")
plt.grid(True)
plt.tight_layout()
plt.show()
print(df['team_type'].value_counts())


# # Sentiment vs Home/Away and crowd
# sns.boxplot(data=df, x='team_type', y='custom_sentiment_score')
# plt.title("Sentiment by Home vs Away")
# plt.grid(True)
# plt.tight_layout()
# plt.show()

# Convert crowd size from string with commas to numeric
df['crowd_numeric'] = df['crowd'].str.replace(',', '').astype(float)

# Create scatter plot with regression line
sns.lmplot(data=df, x='custom_sentiment_score', y='crowd_numeric', height=5, aspect=1.5)
plt.title("Sentiment Score vs Crowd Size")
plt.tight_layout()
plt.show()

In [ ]:
# # Sentiment score by fielding position
# plt.figure(figsize=(10, 5))
# sns.boxplot(data=df, x='수비위치', y='custom_sentiment_score')
# plt.title("Sentiment Score by Fielding Position")
# plt.grid(True)
# plt.tight_layout()
# plt.show()

import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager as fm
import matplotlib as mpl

# 設定支援韓文字體
mpl.rcParams['font.family'] = 'AppleGothic'
mpl.rcParams['axes.unicode_minus'] = False 

plt.figure(figsize=(18, 6))
sns.boxplot(data=df, x='수비위치', y='custom_sentiment_score')
plt.title("Sentiment Score by Fielding Position")
plt.xticks(rotation=45)  # 旋轉 x 軸標籤
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Compute Z-score for normalization
df['sentiment_zscore'] = zscore(df['custom_sentiment_score'])
print("\nZ-scored sentiment added to dataframe as `sentiment_zscore`")

## average performance by players

## statistics analysis

某位選手在某隊的整體表現彙總（跨該隊出賽場次）

In [ ]:
import pandas as pd

# 讀取 CSV
df = pd.read_csv("with_sentiment_boost_rounded.csv")

# 數值欄位轉換
df['crowd'] = df['crowd'].astype(str).str.replace(",", "").astype(float)

# 過濾：有效打數且有情緒分數
df = df[df['타수'] > 0].dropna(subset=['custom_sentiment_score'])

# 建立「選手 + 所屬隊伍」彙總
player_team_summary = df.groupby(['선수명', 'team_name']).agg(
    total_hits=('안타', 'sum'),
    total_at_bats=('타수', 'sum'),
    mean_sentiment_score=('custom_sentiment_score', 'mean'),
    mean_crowd=('crowd', 'mean'),
    home_games=('team_type', lambda x: (x == 'home').sum()),
    total_games=('gameId', 'count'),
    reuse_flag_max=('reuse_flag', 'max'),
    inherit_flag_max=('inherit_flag', 'max'),
    shared_player_max=('shared_player_count', 'max')
).reset_index()

# 計算打擊率與主場比率（保留兩位小數）
player_team_summary['batting_average'] = (player_team_summary['total_hits'] / player_team_summary['total_at_bats']).round(3)
player_team_summary['mean_sentiment_score'] = player_team_summary['mean_sentiment_score'].round(2)
player_team_summary['mean_crowd'] = player_team_summary['mean_crowd'].round(2)
player_team_summary['home_game_ratio'] = (player_team_summary['home_games'] / player_team_summary['total_games']).round(2)

# 選擇需要的欄位
selected_columns = [
    '선수명', 'team_name', 'total_hits', 'total_at_bats', 'batting_average',
    'mean_sentiment_score', 'mean_crowd', 'home_game_ratio',
    'reuse_flag_max', 'inherit_flag_max', 'shared_player_max'
]
player_team_summary = player_team_summary[selected_columns]

# 儲存結果
player_team_summary.to_csv("player_team_summary.csv", index=False, encoding="utf-8-sig")
# Display the summary
print(player_team_summary.head())


H1: Do players perform better when their walk-up lyrics are more emotionally intense?

In [ ]:
import pandas as pd
from scipy.stats import pearsonr, spearmanr
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler

# 讀取 player summary（球員-球隊層級）
df = pd.read_csv("player_team_summary.csv")

# 檢查缺漏值並濾除
df = df.dropna(subset=['mean_sentiment_score', 'batting_average'])

# 相關分析
pearson_r, pearson_p = pearsonr(df['mean_sentiment_score'], df['batting_average'])
spearman_r, spearman_p = spearmanr(df['mean_sentiment_score'], df['batting_average'])

# 線性回歸（標準化）
scaler = StandardScaler()
df['z_sentiment'] = scaler.fit_transform(df[['mean_sentiment_score']])

model = smf.ols('batting_average ~ z_sentiment', data=df).fit()

# 輸出結果
print("H1: Emotionally intense lyrics and batting performance\n")
print(f"Pearson correlation: r = {pearson_r:.3f}, p = {pearson_p:.4f}")
print(f"Spearman correlation: ρ = {spearman_r:.3f}, p = {spearman_p:.4f}")
print("\n--- Linear Regression Summary ---")
print(model.summary())


分組 t-test & Boxplot（高 vs 低情緒）

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

# 讀入資料
df = pd.read_csv("player_team_summary.csv")
df = df.dropna(subset=['mean_sentiment_score', 'batting_average'])

# 建立高低情緒分組（中位數分割）
median_score = df['mean_sentiment_score'].median()
df['sentiment_group'] = df['mean_sentiment_score'].apply(lambda x: 'High' if x > median_score else 'Low')

# t-test
group_high = df[df['sentiment_group'] == 'High']['batting_average']
group_low = df[df['sentiment_group'] == 'Low']['batting_average']
t_stat, p_val = ttest_ind(group_high, group_low)

print(f"t-test result: t = {t_stat:.3f}, p = {p_val:.4f}")

# Boxplot
plt.figure(figsize=(6, 4))
sns.boxplot(data=df, x='sentiment_group', y='batting_average', palette='Set2')
plt.title("Batting Average by Sentiment Group")
plt.xlabel("Sentiment Group")
plt.ylabel("Batting Average")
plt.grid(True)
plt.tight_layout()
plt.show()


加入控制變項的多變項回歸（控制觀眾、主場比例）

In [ ]:
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler

# 標準化數值欄位
scaler = StandardScaler()
df[['z_sentiment', 'z_crowd', 'z_home_ratio']] = scaler.fit_transform(
    df[['mean_sentiment_score', 'mean_crowd', 'home_game_ratio']]
)

# 多變項迴歸
model = smf.ols('batting_average ~ z_sentiment + z_crowd + z_home_ratio', data=df).fit()
print(model.summary())

# 檢查模型的假設
import statsmodels.api as sm
# 殘差分析
residuals = model.resid
sm.qqplot(residuals, line='s')
plt.title("QQ Plot of Residuals")
plt.show()
# 殘差與預測值的散佈圖
plt.figure(figsize=(6, 4))
sns.scatterplot(x=model.fittedvalues, y=residuals)
plt.axhline(0, color='red', linestyle='--')
plt.title("Residuals vs Fitted Values")
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.grid(True)
plt.tight_layout()
plt.show()
# 檢查多重共線性
from statsmodels.stats.outliers_influence import variance_inflation_factor
# 計算 VIF
X = df[['z_sentiment', 'z_crowd', 'z_home_ratio']]
vif_data = pd.DataFrame()
vif_data['feature'] = X.columns
vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print("\nVariance Inflation Factor (VIF):")
print(vif_data)
# 檢查異方差性
from statsmodels.stats.diagnostic import het_breuschpagan
# Breusch-Pagan test
bp_test = het_breuschpagan(residuals, model.model.exog)
print("\nBreusch-Pagan test result:")
print(f"LM Statistic: {bp_test[0]:.3f}, p-value: {bp_test[1]:.4f}")
# 檢查自相關
from statsmodels.stats.diagnostic import acorr_ljungbox
# Ljung-Box test
ljung_box_test = acorr_ljungbox(residuals, lags=[10], return_df=True)
print("\nLjung-Box test result:")
print(ljung_box_test)
# Save the model summary to a text file
with open("regression_model_summary.txt", "w") as f:
    f.write(model.summary().as_text())
# Save the VIF data to a CSV file
vif_data.to_csv("vif_data.csv", index=False)

In [ ]:
plt.figure(figsize=(6, 4))
sns.regplot(data=df, x='mean_sentiment_score', y='batting_average', scatter_kws={'alpha':0.6})
plt.title("Batting Average vs. Mean Sentiment Score")
plt.xlabel("Mean Sentiment Score")
plt.ylabel("Batting Average")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# 使用 robust 標準誤進行 OLS 回歸（補救異方差）
robust_model = smf.ols('batting_average ~ z_sentiment + z_crowd + z_home_ratio', data=df).fit(cov_type='HC3')
print(robust_model.summary())

H2: The lyrical effect becomes stronger at home, where collective cheering happens.

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler

# 載入資料
df = pd.read_csv("with_sentiment_boost_rounded.csv")

# 確保欄位轉為數值（如果有字串形式的 0）
df['타수'] = pd.to_numeric(df['타수'], errors='coerce')
df['타율'] = pd.to_numeric(df['타율'], errors='coerce')
df['custom_sentiment_score'] = pd.to_numeric(df['custom_sentiment_score'], errors='coerce')

# 保留有打席的場次（去除 타수 = 0）
df = df[df['타수'] > 0].copy()

# 建立主場欄位
df['is_home'] = df['team_type'].apply(lambda x: 1 if x == 'home' else 0)

# 標準化情緒分數
scaler = StandardScaler()
df['z_sentiment'] = scaler.fit_transform(df[['custom_sentiment_score']])

# 建立交互作用項（情緒分數 x 主場）
df['interaction'] = df['z_sentiment'] * df['is_home']

# 執行 robust OLS 回歸（HC3 robust 標準誤）
model = smf.ols('타율 ~ z_sentiment + is_home + interaction', data=df).fit(cov_type='HC3')

# 顯示模型摘要
print(model.summary())


Mixed Effects Model

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler

# 載入原始逐場資料
df = pd.read_csv("with_sentiment_boost_rounded.csv")
df = df[df['타수'] > 0].copy()
df['is_home'] = df['team_type'].apply(lambda x: 1 if x == 'home' else 0)

# 標準化
scaler = StandardScaler()
df['z_sentiment'] = scaler.fit_transform(df[['custom_sentiment_score']])

# Mixed Effects Model (控制球員為隨機效果)
import statsmodels.api as sm
model = smf.mixedlm("타율 ~ z_sentiment + is_home + z_sentiment:is_home",
                    data=df,
                    groups=df["선수명"]).fit(reml=False)
print(model.summary())


H3: Does a larger infield crowd further amplify the motivational impact of lyrics?

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
import statsmodels.formula.api as smf

# 讀取資料
df = pd.read_csv("with_sentiment_boost_rounded.csv")

# 過濾無效資料
df = df[df['타수'] > 0].dropna(subset=['custom_sentiment_score', '타율', 'crowd'])

# 處理 crowd 欄位中的逗號，轉為 float
df['crowd'] = pd.to_numeric(df['crowd'].astype(str).str.replace(",", ""), errors='coerce')

# 標準化欄位
scaler = StandardScaler()
df[['z_sentiment', 'z_crowd']] = scaler.fit_transform(df[['custom_sentiment_score', 'crowd']])

# Mixed Effects Model：以球員為隨機效果，觀察情緒分數與觀眾交互作用
model_h3 = smf.mixedlm("타율 ~ z_sentiment * z_crowd", df, groups=df["선수명"])
result_h3 = model_h3.fit(method='lbfgs')  # 用 lbfgs 解決大型資料較穩定
print(result_h3.summary())


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# 載入資料
df = pd.read_csv("with_sentiment_boost_rounded.csv")

# 數值轉換（處理 crowd 的格式）
df['crowd'] = df['crowd'].astype(str).str.replace(',', '').astype(float)

# 篩選條件：有有效打數且有情緒分數
df = df[(df['타수'] > 0) & df['custom_sentiment_score'].notna()]

# 新增打擊率欄位
df['batting_average'] = df['안타'] / df['타수']

# 標準化變項
scaler = StandardScaler()
df[['z_sentiment', 'z_crowd']] = scaler.fit_transform(df[['custom_sentiment_score', 'crowd']])

# 分群：觀眾數高低（以中位數為界）
median_crowd = df['z_crowd'].median()
df['crowd_group'] = df['z_crowd'].apply(lambda x: 'High Crowd' if x >= median_crowd else 'Low Crowd')

# 畫圖
plt.figure(figsize=(8, 5))
sns.regplot(data=df[df['crowd_group'] == 'High Crowd'], x='z_sentiment', y='batting_average', label='High Crowd', scatter_kws={'alpha':0.3})
sns.regplot(data=df[df['crowd_group'] == 'Low Crowd'], x='z_sentiment', y='batting_average', label='Low Crowd', scatter_kws={'alpha':0.3})
plt.title("Interaction: Sentiment vs Batting Average by Crowd Size")
plt.xlabel("Standardized Sentiment Score (z)")
plt.ylabel("Batting Average")
plt.legend()
plt.tight_layout()
plt.grid(True)
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import statsmodels.formula.api as smf

# 1. 載入資料
df = pd.read_csv("with_sentiment_boost_rounded.csv")
df['crowd'] = df['crowd'].astype(str).str.replace(',', '', regex=False).astype(float)

# 2. 篩選條件：有打數、有情緒分數
df = df[(df['타수'] > 0) & df['custom_sentiment_score'].notna()]

# 3. 先算出每位球員的總安打、總打數、平均情緒分數與觀眾人數
player_avg = (
    df.groupby('선수명')
    .agg(
        total_hits=('안타', 'sum'),
        total_at_bats=('타수', 'sum'),
        avg_sentiment=('custom_sentiment_score', 'mean'),
        avg_crowd=('crowd', 'mean')
    )
    .reset_index()
)

# 4. 計算打擊率
player_avg['batting_average'] = player_avg['total_hits'] / player_avg['total_at_bats']

# 5. 標準化
scaler = StandardScaler()
player_avg[['z_sentiment', 'z_crowd']] = scaler.fit_transform(player_avg[['avg_sentiment', 'avg_crowd']])

# 6. 分群（依平均觀眾數中位數）
median_crowd = player_avg['z_crowd'].median()
player_avg['crowd_group'] = player_avg['z_crowd'].apply(lambda x: 'High Crowd' if x >= median_crowd else 'Low Crowd')

# 7. 畫圖（依群組分色）
plt.figure(figsize=(8, 5))
sns.regplot(data=player_avg[player_avg['crowd_group'] == 'High Crowd'],
            x='z_sentiment', y='batting_average', label='High Crowd',
            scatter_kws={'alpha': 0.5})
sns.regplot(data=player_avg[player_avg['crowd_group'] == 'Low Crowd'],
            x='z_sentiment', y='batting_average', label='Low Crowd',
            scatter_kws={'alpha': 0.5})
plt.title("Interaction: Sentiment vs Batting Average (Per Player Avg) by Crowd Size")
plt.xlabel("Standardized Avg Sentiment Score")
plt.ylabel("Batting Average (Total Hits / Total AB)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# 8. 回歸模型（交互作用）
player_avg['crowd_group'] = player_avg['crowd_group'].astype('category')
model = smf.ols('batting_average ~ z_sentiment * crowd_group', data=player_avg).fit()
print(model.summary())


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# 讀入球員平均資料
df = pd.read_csv("player_team_summary.csv")

# 標準化
scaler = StandardScaler()
df[['z_sentiment', 'z_crowd']] = scaler.fit_transform(df[['mean_sentiment_score', 'mean_crowd']])

# 分群（觀眾中位數）
median_crowd = df['z_crowd'].median()
df['crowd_group'] = df['z_crowd'].apply(lambda x: 'High Crowd' if x >= median_crowd else 'Low Crowd')

# 畫圖
plt.figure(figsize=(8, 5))
sns.regplot(data=df[df['crowd_group'] == 'High Crowd'], x='z_sentiment', y='batting_average', label='High Crowd', scatter_kws={'alpha':0.5})
sns.regplot(data=df[df['crowd_group'] == 'Low Crowd'], x='z_sentiment', y='batting_average', label='Low Crowd', scatter_kws={'alpha':0.5})
plt.title("Average Batting vs Sentiment (Grouped by Crowd Level)")
plt.xlabel("Standardized Sentiment Score (z)")
plt.ylabel("Average Batting Average")
plt.legend()
plt.tight_layout()
plt.grid(True)
plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 載入資料
df = pd.read_csv("with_sentiment_boost_rounded.csv")

# 前處理
df['타수'] = pd.to_numeric(df['타수'], errors='coerce')
df['타율'] = pd.to_numeric(df['타율'], errors='coerce')
df = df[(df['타수'] > 0) & df['custom_sentiment_score'].notna() & df['타율'].notna()]
df = df[df['team_type'].isin(['home', 'away'])]

# 分組（以中位數分為 High / Low sentiment）
median_sentiment = df['custom_sentiment_score'].median()
df['sentiment_group'] = df['custom_sentiment_score'].apply(lambda x: 'High' if x >= median_sentiment else 'Low')

# 合併標籤：主/客場 + 情緒分數組合
df['group'] = df['team_type'].str.capitalize() + ' / ' + df['sentiment_group']

# Boxplot 畫圖
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='group', y='타율', palette='Set2')
plt.title("Batting Average by Home/Away × Sentiment Group")
plt.xlabel("Game Context (Home/Away × Sentiment)")
plt.ylabel("Batting Average")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 載入資料
df = pd.read_csv("with_sentiment_boost_rounded.csv")

# 前處理
df['타수'] = pd.to_numeric(df['타수'], errors='coerce')
df['타율'] = pd.to_numeric(df['타율'], errors='coerce')
df = df[(df['타수'] > 0) & df['custom_sentiment_score'].notna() & df['타율'].notna()]
df = df[df['team_type'].isin(['home', 'away'])]

# 計算每位球員的平均打擊率與平均情緒分數（分主場與客場分開看）
player_avg = (
    df.groupby(['선수명', 'team_type'])[['타율', 'custom_sentiment_score']]
    .mean()
    .reset_index()
)

# 分組：以中位數區分 High / Low sentiment
median_sentiment = player_avg['custom_sentiment_score'].median()
player_avg['sentiment_group'] = player_avg['custom_sentiment_score'].apply(lambda x: 'High' if x >= median_sentiment else 'Low')

# 合併標籤：主/客場 + 情緒分數組合
player_avg['group'] = player_avg['team_type'].str.capitalize() + ' / ' + player_avg['sentiment_group']

# Boxplot 畫圖
plt.figure(figsize=(8, 5))
sns.boxplot(data=player_avg, x='group', y='타율', palette='Set2')
plt.title("Player Avg Batting Average by Home/Away × Sentiment Group")
plt.xlabel("Game Context (Home/Away × Sentiment)")
plt.ylabel("Average Batting Average (per player)")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.formula.api import ols
import scipy.stats as stats

# 讀取資料
df = pd.read_csv("with_sentiment_boost_rounded.csv")

# 前處理
df['타수'] = pd.to_numeric(df['타수'], errors='coerce')
df['타율'] = pd.to_numeric(df['타율'], errors='coerce')
df = df[(df['타수'] > 0) & df['custom_sentiment_score'].notna() & df['타율'].notna()]
df = df[df['team_type'].isin(['home', 'away'])]

# 每位球員在主/客場的平均打擊率與情緒分數
player_avg = (
    df.groupby(['선수명', 'team_type'])[['타율', 'custom_sentiment_score']]
    .mean()
    .reset_index()
)

# 分組（依情緒中位數區分）
median_sentiment = player_avg['custom_sentiment_score'].median()
player_avg['sentiment_group'] = player_avg['custom_sentiment_score'].apply(
    lambda x: 'High' if x >= median_sentiment else 'Low'
)

# 合併分組標籤供可視化使用
player_avg['group'] = player_avg['team_type'].str.capitalize() + ' / ' + player_avg['sentiment_group']

# -------- 🎯 Two-way ANOVA -------- #
model = ols('타율 ~ C(team_type) + C(sentiment_group) + C(team_type):C(sentiment_group)', data=player_avg).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
print("🔍 ANOVA 結果：\n", anova_table)

# -------- 📊 Boxplot 視覺化 -------- #
plt.figure(figsize=(8, 5))
sns.boxplot(data=player_avg, x='group', y='타율', palette='Set2')
plt.title("Player Avg Batting Average by Home/Away × Sentiment Group")
plt.xlabel("Game Context (Home/Away × Sentiment)")
plt.ylabel("Average Batting Average (per player)")
plt.grid(True)
plt.tight_layout()
plt.show()


H4：歌詞在對上較強對手時是否更有影響力？

In [ ]:
import pandas as pd

# 讀取資料
df = pd.read_csv("with_sentiment_boost_rounded.csv")

# 將得分欄位轉為數值
df['득점'] = pd.to_numeric(df['득점'], errors='coerce')

# 計算每場比賽每支球隊的得分
game_team_score = df.groupby(['gameId', 'team_name'])['득점'].sum().reset_index(name='team_score')

# 合併得分資訊回原始資料
df = pd.merge(df, game_team_score, on=['gameId', 'team_name'], how='left')

# 找出該場比賽的對手與對手得分
opponent_score = game_team_score.rename(columns={
    'team_name': 'opponent_name',
    'team_score': 'opponent_score'
})
df = pd.merge(df, opponent_score, on='gameId', how='left')
df = df[df['team_name'] != df['opponent_name']]  # 避免合併到自己

# 判斷該隊是否勝利
df['is_win'] = df['team_score'] > df['opponent_score']

# 計算每隊勝率
team_winrate = df.groupby('team_name')['is_win'].mean().reset_index(name='winrate')

# 合併對手勝率到主資料
df = pd.merge(df, team_winrate.rename(columns={'team_name': 'opponent_name', 'winrate': 'opponent_winrate'}),
              on='opponent_name', how='left')

# 儲存新檔案
df.to_csv("with_opponent_winrate.csv", index=False, encoding="utf-8-sig")


In [ ]:
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler

# 讀取資料
df = pd.read_csv("with_opponent_winrate.csv")

# 標準化欄位
scaler = StandardScaler()
df[['z_sentiment', 'z_opponent_winrate']] = scaler.fit_transform(
    df[['custom_sentiment_score', 'opponent_winrate']]
)

# 建立交互作用項
df['interaction'] = df['z_sentiment'] * df['z_opponent_winrate']

# Mixed Linear Model: 對球員做隨機效果控制
import statsmodels.api as sm
from statsmodels.regression.mixed_linear_model import MixedLM

model = MixedLM.from_formula(
    "타율 ~ z_sentiment + z_opponent_winrate + interaction",
    groups="선수명", data=df
)
result = model.fit(method='lbfgs')
print(result.summary())


H5: H5 / H5b / H5c（重複、繼承、共用的歌詞是否影響效果）

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy.stats import ttest_ind

# Load data
df = pd.read_csv("player_team_summary.csv")

# -----------------------------
# H6: Reuse vs Batting Average
# -----------------------------
plt.figure(figsize=(6, 4))
sns.boxplot(data=df, x='reuse_flag_max', y='batting_average')
plt.title('H6: Batting Average by Reuse Flag')
plt.xlabel('Reuse Flag (0 = No, 1 = Yes)')
plt.ylabel('Batting Average')
plt.tight_layout()
plt.show()

# T-test
reuse0 = df[df['reuse_flag_max'] == 0]['batting_average']
reuse1 = df[df['reuse_flag_max'] == 1]['batting_average']
t_reuse, p_reuse = ttest_ind(reuse0, reuse1, equal_var=False)
print(f"H6 - Reuse Flag: t = {t_reuse:.4f}, p = {p_reuse:.4f}")

# -----------------------------
# H6b: Inherit vs Batting Average
# -----------------------------
plt.figure(figsize=(6, 4))
sns.boxplot(data=df, x='inherit_flag_max', y='batting_average')
plt.title('H6b: Batting Average by Inherit Flag')
plt.xlabel('Inherit Flag (0 = No, 1 = Yes)')
plt.ylabel('Batting Average')
plt.tight_layout()
plt.show()

# T-test
inh0 = df[df['inherit_flag_max'] == 0]['batting_average']
inh1 = df[df['inherit_flag_max'] == 1]['batting_average']
t_inh, p_inh = ttest_ind(inh0, inh1, equal_var=False)
print(f"H6b - Inherit Flag: t = {t_inh:.4f}, p = {p_inh:.4f}")

# -----------------------------
# H6c: Shared Count vs Batting Average (Regression)
# -----------------------------
model = smf.ols('batting_average ~ shared_player_max', data=df).fit()
print(model.summary())

plt.figure(figsize=(6, 4))
sns.regplot(data=df, x='shared_player_max', y='batting_average', scatter_kws={'alpha':0.5})
plt.title('H6c: Shared Player Count vs Batting Average')
plt.xlabel('Shared Player Count')
plt.ylabel('Batting Average')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Load data
df = pd.read_csv("player_team_summary.csv")

# Group by shared_player_count
grouped = df.groupby('shared_player_max')['batting_average'].agg(['mean', 'count', 'std']).reset_index()
grouped['sem'] = grouped['std'] / grouped['count']**0.5  # Standard Error of the Mean

# Plot
plt.figure(figsize=(8, 5))
sns.pointplot(
    data=grouped,
    x='shared_player_max',
    y='mean',
    join=True,
    capsize=0.2,
    errwidth=1.5,
    color='steelblue'
)
plt.title("Effect of Shared Song Count on Batting Average")
plt.xlabel("Number of Players Sharing the Same Song")
plt.ylabel("Average Batting Average")
plt.grid(True, axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

# 載入 summary 資料（含共用資訊）
df = pd.read_csv("player_team_summary.csv")

# 去除缺漏值
df = df[df['shared_player_max'].notna() & df['batting_average'].notna()]

# 建立高共用 / 低共用 分組
df['shared_group'] = df['shared_player_max'].apply(lambda x: 'High Share (>=3)' if x >= 3 else 'Low Share (<3)')

# t-test 比較兩組打擊表現
high = df[df['shared_group'] == 'High Share (>=3)']['batting_average']
low = df[df['shared_group'] == 'Low Share (<3)']['batting_average']
t_stat, p_val = ttest_ind(high, low)
print(f"T-test: t = {t_stat:.4f}, p = {p_val:.4f}")

# 視覺化
plt.figure(figsize=(6, 5))
sns.boxplot(data=df, x='shared_group', y='batting_average', palette="Set2")
plt.title('Batting Performance by Song Sharing Level')
plt.ylabel('Batting Average')
plt.xlabel('Song Sharing Group')
plt.tight_layout()
plt.show()


H9: Teams differ in lyrical preference (sentiment score).
Are some teams using more emotionally intense songs than others?

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Load data
df = pd.read_csv("with_sentiment_boost_rounded.csv")

# Clean: drop rows with missing sentiment or team info
df = df[df['custom_sentiment_score'].notna() & df['team_name'].notna()]

# ANOVA: Check if team means differ
model = ols('custom_sentiment_score ~ C(team_name)', data=df).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
print(anova_table)

# Boxplot: Compare distribution across teams
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x='team_name', y='custom_sentiment_score')
plt.xticks(rotation=45)
plt.title('Team-wise Distribution of Fight Song Sentiment Scores')
plt.xlabel('Team')
plt.ylabel('Custom Sentiment Score')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import matplotlib.pyplot as plt

# 確保資料中 team_name 與 custom_sentiment_score 有效
df = df[df['custom_sentiment_score'].notna() & df['team_name'].notna()]

# 執行 Tukey HSD
tukey = pairwise_tukeyhsd(
    endog=df['custom_sentiment_score'],
    groups=df['team_name'],
    alpha=0.05
)

# 顯示結果
print(tukey)

# 可視化 Tukey HSD
tukey.plot_simultaneous(figsize=(12, 6))
plt.title('Tukey HSD: Team Differences in Sentiment Score')
plt.xlabel('Mean Difference in Sentiment Score')
plt.grid(True)
plt.show()


In [ ]:
import pandas as pd
from collections import defaultdict
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# 讀取資料
df = pd.read_csv("with_sentiment_boost_rounded.csv")

# 排除沒有歌詞（"none"）的資料
df = df.dropna(subset=['team_name', '선수명', 'Song_lyrics'])
df = df[df['Song_lyrics'].str.lower() != 'none']

# 避免重複：每位球員每首歌詞只計算一次
df_unique = df.drop_duplicates(subset=['team_name', '선수명', 'Song_lyrics'])

# 彙總每支球隊的歌詞
team_lyrics = defaultdict(str)
for _, row in df_unique.iterrows():
    team_lyrics[row['team_name']] += row['Song_lyrics'] + " "

# 自動畫出所有球隊的詞雲
for team, lyrics in team_lyrics.items():
    wordcloud = WordCloud(
        width=800,
        height=400,
        background_color='white',
        font_path='AppleGothic'
    ).generate(lyrics)

    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(f'Word Cloud of Fight Songs: {team}')
    plt.tight_layout()
    plt.show()



In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt

# 1. 讀取 CSV 檔案
df = pd.read_csv("with_sentiment_boost_rounded.csv")

# 2. 過濾不必要的欄位與資料
df = df.dropna(subset=['team_name', '선수명', 'Song_lyrics'])
df = df[df['Song_lyrics'].str.lower() != 'none']
df_unique = df.drop_duplicates(subset=['team_name', '선수명', 'Song_lyrics'])

# 3. 建立每隊歌詞全集
team_texts = df_unique.groupby('team_name')['Song_lyrics'].apply(lambda x: ' '.join(x)).reset_index()

# 4. 建立 TF-IDF 模型
vectorizer = TfidfVectorizer(max_features=1000)
X = vectorizer.fit_transform(team_texts['Song_lyrics'])
feature_names = vectorizer.get_feature_names_out()

# 5. 對每支球隊單獨畫圖
for i, row in team_texts.iterrows():
    team = row['team_name']
    lyrics = row['Song_lyrics']
    tfidf_vector = X[i].toarray().flatten()
    top_indices = tfidf_vector.argsort()[::-1][:10]
    top_words = [feature_names[idx] for idx in top_indices]
    top_scores = [tfidf_vector[idx] for idx in top_indices]

    # 視覺化
    plt.figure(figsize=(8, 5))
    plt.barh(top_words[::-1], top_scores[::-1], color='skyblue')
    plt.title(f"Top TF-IDF Words for {team}")
    plt.xlabel("TF-IDF Score")
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 讀入資料
df = pd.read_csv("with_sentiment_boost_rounded.csv")

# 前處理：只保留有有效值的行
df = df[(df['타수'] > 0) & df['custom_sentiment_score'].notna()]
df['batting_average'] = df['안타'] / df['타수']

# 各隊平均情緒與打擊率
team_stats = df.groupby('team_name').agg(
    avg_sentiment=('custom_sentiment_score', 'mean'),
    avg_batting_avg=('batting_average', 'mean'),
    player_count=('선수명', 'nunique')
).reset_index()

# 視覺化：每隊打擊率 vs 歌詞情緒
plt.figure(figsize=(10,6))
plt.scatter(team_stats['avg_sentiment'], team_stats['avg_batting_avg'], s=team_stats['player_count']*10, alpha=0.7)

# 標籤
for _, row in team_stats.iterrows():
    plt.text(row['avg_sentiment']+0.01, row['avg_batting_avg'], row['team_name'], fontsize=9)

plt.title("Average Fight Song Sentiment vs Batting Average (by Team)")
plt.xlabel("Average Sentiment Score")
plt.ylabel("Average Batting Average")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import seaborn as sns
import matplotlib.pyplot as plt

# 載入資料
df = pd.read_csv("with_sentiment_boost_rounded.csv")

# 篩選出有效歌詞資料
df = df[df['Song_lyrics'].notna() & (df['Song_lyrics'] != "none")]

# 移除同一球員多場重複出現：只保留每位球員第一首歌
df_unique = df.drop_duplicates(subset=['선수명', 'Song_lyrics'])

# 建立 team → 歌詞字串的映射（每人一次，不重複）
team_lyrics = df_unique.groupby('team_name')['Song_lyrics'].apply(lambda x: ' '.join(x)).to_dict()

# 向量化：TF-IDF
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(team_lyrics.values())

# 球隊名稱
team_names = list(team_lyrics.keys())

# 計算餘弦相似度
cos_sim = cosine_similarity(tfidf_matrix)

# 建立 DataFrame 方便視覺化
sim_df = pd.DataFrame(cos_sim, index=team_names, columns=team_names)

# 視覺化 Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(sim_df, annot=True, fmt=".2f", cmap="YlGnBu", square=True, linewidths=0.5)
plt.title("Cosine Similarity of Fight Song Lyrics between Teams")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
from kiwipiepy import Kiwi
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

# 載入資料
df = pd.read_csv("with_sentiment_boost_rounded.csv")
df = df[df['Song_lyrics'].notna() & (df['Song_lyrics'] != "none")]
df_unique = df.drop_duplicates(subset=['선수명', 'Song_lyrics'])

# 分組組合歌詞
team_lyrics = df_unique.groupby('team_name')['Song_lyrics'].apply(lambda x: ' '.join(x)).to_dict()

# 詞性標記器
kiwi = Kiwi()

# 詞性標記並統計比重
team_pos_ratio = {}
for team, lyrics in team_lyrics.items():
    tokens = kiwi.tokenize(lyrics)
    pos_counts = Counter([token.tag for token in tokens])
    total = sum(pos_counts.values())
    pos_ratio = {tag: count / total for tag, count in pos_counts.items()}
    team_pos_ratio[team] = pos_ratio

# 轉為 DataFrame，缺失補 0
pos_df = pd.DataFrame(team_pos_ratio).fillna(0).T

# 選前幾個主要詞性
top_tags = pos_df.mean().sort_values(ascending=False).head(6).index
pos_df = pos_df[top_tags]

# 視覺化
plt.figure(figsize=(12, 6))
pos_df.plot(kind='bar', stacked=True, colormap='tab20', figsize=(12, 6))
plt.title("POS Tag Ratio of Fight Song Lyrics by Team")
plt.ylabel("Proportion")
plt.xlabel("Team")
plt.xticks(rotation=45)
plt.legend(title="POS Tag", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
from kiwipiepy import Kiwi
from collections import Counter
import plotly.express as px

# 載入資料
df = pd.read_csv("with_sentiment_boost_rounded.csv")
df = df[df['Song_lyrics'].notna() & (df['Song_lyrics'] != "none")]
df_unique = df.drop_duplicates(subset=['선수명', 'Song_lyrics'])

# 組合每隊歌詞
team_lyrics = df_unique.groupby('team_name')['Song_lyrics'].apply(lambda x: ' '.join(x)).to_dict()

# 初始化 Kiwi
kiwi = Kiwi()

# 統計每隊詞性比例
team_pos_data = []
for team, lyrics in team_lyrics.items():
    tokens = kiwi.tokenize(lyrics)
    pos_counts = Counter([token.tag for token in tokens])
    total = sum(pos_counts.values())
    for tag, count in pos_counts.items():
        team_pos_data.append({
            'team_name': team,
            'pos_tag': tag,
            'ratio': count / total
        })

# 建立 DataFrame 並篩選出出現率最高的前六名 POS
pos_df = pd.DataFrame(team_pos_data)
top_tags = pos_df.groupby('pos_tag')['ratio'].mean().sort_values(ascending=False).head(6).index
filtered_df = pos_df[pos_df['pos_tag'].isin(top_tags)]

# 使用 Plotly 畫 bar chart（grouped，不是 stacked）
fig = px.bar(
    filtered_df,
    x="team_name",
    y="ratio",
    color="pos_tag",
    barmode="group",
    title="POS Tag Ratio by Team (Top 6 Tags)",
    labels={"ratio": "Proportion", "team_name": "Team", "pos_tag": "POS Tag"}
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()


In [ ]:
import pandas as pd
from transformers import BertTokenizer, BertModel
import torch
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# 載入資料
df = pd.read_csv("with_sentiment_boost_rounded.csv")
df = df[df['Song_lyrics'].notna() & (df['Song_lyrics'] != "none")]
df_unique = df.drop_duplicates(subset=['선수명', 'Song_lyrics'])

# 合併每隊歌詞
team_lyrics = df_unique.groupby('team_name')['Song_lyrics'].apply(lambda x: ' '.join(x)).to_dict()
teams = list(team_lyrics.keys())
texts = list(team_lyrics.values())

# 載入 BERT 模型
tokenizer = BertTokenizer.from_pretrained("klue/bert-base")
model = BertModel.from_pretrained("klue/bert-base")
model.eval()

# 嵌入向量
embeddings = []
with torch.no_grad():
    for text in texts:
        inputs = tokenizer(text[:512], return_tensors="pt", truncation=True, max_length=512)
        outputs = model(**inputs)
        cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze().numpy()
        embeddings.append(cls_embedding)

# 聚類與降維（改為 5 clusters）
kmeans = KMeans(n_clusters=3, random_state=42).fit(embeddings)
pca = PCA(n_components=2)
points = pca.fit_transform(embeddings)

# 畫圖
plt.figure(figsize=(8, 6))
for i, (x, y) in enumerate(points):
    plt.scatter(x, y, c=f"C{kmeans.labels_[i]}", label=teams[i], s=100)
    plt.text(x + 0.01, y + 0.01, teams[i], fontsize=9)
plt.title("Team Fight Song Clustering (BERT + PCA, 3 Clusters)")
plt.xlabel("PCA1")
plt.ylabel("PCA2")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()



In [ ]:
import pandas as pd
from transformers import BertTokenizer, BertModel
import torch
from sklearn.cluster import KMeans
import umap
import matplotlib.pyplot as plt
import seaborn as sns

# 字體設定（mac: AppleGothic, Windows: Malgun Gothic）
plt.rcParams['font.family'] = 'AppleGothic'

# 載入資料
df = pd.read_csv("with_sentiment_boost_rounded.csv")
df = df[df['Song_lyrics'].notna() & (df['Song_lyrics'] != "none")]
df_unique = df.drop_duplicates(subset=['선수명', 'Song_lyrics'])

# 合併每隊歌詞
team_lyrics = df_unique.groupby('team_name')['Song_lyrics'].apply(lambda x: ' '.join(x)).to_dict()
teams = list(team_lyrics.keys())
texts = list(team_lyrics.values())

# 載入 BERT 模型
tokenizer = BertTokenizer.from_pretrained("klue/bert-base")
model = BertModel.from_pretrained("klue/bert-base")
model.eval()

# 取得 BERT 嵌入
embeddings = []
with torch.no_grad():
    for text in texts:
        inputs = tokenizer(text[:512], return_tensors="pt", truncation=True, max_length=512)
        outputs = model(**inputs)
        cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze().numpy()
        embeddings.append(cls_embedding)

# KMeans 分群（5 clusters）
kmeans = KMeans(n_clusters=3, random_state=42)
labels = kmeans.fit_predict(embeddings)

# UMAP 降維
reducer = umap.UMAP(n_components=2, random_state=42)
umap_coords = reducer.fit_transform(embeddings)

# 整理 dataframe
df_plot = pd.DataFrame(umap_coords, columns=['UMAP1', 'UMAP2'])
df_plot['team_name'] = teams
df_plot['cluster'] = labels

# 畫圖
plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=df_plot,
    x='UMAP1',
    y='UMAP2',
    hue='cluster',
    style='team_name',
    palette='Set1',
    s=120
)

for i, row in df_plot.iterrows():
    plt.text(row['UMAP1'] + 0.1, row['UMAP2'], row['team_name'], fontsize=9)

plt.title("UMAP: Team Fight Song Clustering (BERT + KMeans, 3 Clusters)")
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import umap
from sklearn.cluster import KMeans
import plotly.express as px

# 讀取資料
df = pd.read_csv("with_sentiment_boost_rounded.csv")

# 篩選有效歌詞資料
df = df[df['Song_lyrics'].notna() & (df['Song_lyrics'] != "none")].copy()
df_unique = df.drop_duplicates(subset=["team_name", "Song_lyrics"]).copy()

# 嵌入歌詞向量
model = SentenceTransformer('distiluse-base-multilingual-cased-v2')
embeddings = model.encode(df_unique["Song_lyrics"].tolist(), show_progress_bar=True)

# UMAP 降維
reducer = umap.UMAP(n_components=2, random_state=42)
embedding_2d = reducer.fit_transform(embeddings)
df_unique['umap_x'] = embedding_2d[:, 0]
df_unique['umap_y'] = embedding_2d[:, 1]

# 加入聚類
kmeans = KMeans(n_clusters=4, random_state=42)
df_unique['cluster'] = kmeans.fit_predict(embedding_2d)

# 製作 hover 字串
df_unique['hover'] = (
    "Team: " + df_unique['team_name'] +
    "<br>Player: " + df_unique['선수명'] +
    "<br>Sentiment: " + df_unique['custom_sentiment_score'].astype(str) +
    "<br>Lyrics: " + df_unique['Song_lyrics'].str.slice(0, 50) + "..."
)

# 畫 Plotly 散點圖
fig = px.scatter(
    df_unique,
    x='umap_x',
    y='umap_y',
    color='team_name',
    size='custom_sentiment_score',
    hover_name='선수명',
    hover_data={'umap_x': False, 'umap_y': False, 'custom_sentiment_score': False, 'cluster': True, 'hover': True},
    title="UMAP Visualization of Fight Song Lyrics (BERT Embeddings)",
    size_max=40,
    template='plotly_white'
)

fig.update_traces(marker=dict(line=dict(width=0.5, color='DarkSlateGrey')))
fig.update_layout(legend_title_text='Team', hovermode='closest')
fig.show()


In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import umap
from sklearn.cluster import KMeans
import plotly.express as px

# 讀取資料
df = pd.read_csv("with_sentiment_boost_rounded.csv")

# 篩選有效歌詞
df = df[df['Song_lyrics'].notna() & (df['Song_lyrics'] != "none")].copy()

# 計算打擊率
df['batting_avg'] = df['안타'] / df['타수']
df = df[df['batting_avg'].notna() & df['custom_sentiment_score'].notna()]

# 每隊平均打擊率與情緒分數
team_avg = df.groupby("team_name").agg(
    avg_batting_avg=('batting_avg', 'mean'),
    avg_sentiment=('custom_sentiment_score', 'mean')
).reset_index()

# 避免重複計算歌詞：取每隊唯一歌詞（或每首歌只保留一筆）
df_unique = df.drop_duplicates(subset=["team_name", "Song_lyrics"]).copy()

# 合併團隊平均表現
df_unique = pd.merge(df_unique, team_avg, on="team_name", how="left")

# BERT 嵌入
model = SentenceTransformer('distiluse-base-multilingual-cased-v2')
embeddings = model.encode(df_unique["Song_lyrics"].tolist(), show_progress_bar=True)

# UMAP 降維
reducer = umap.UMAP(n_components=2, random_state=42)
embedding_2d = reducer.fit_transform(embeddings)
df_unique['umap_x'] = embedding_2d[:, 0]
df_unique['umap_y'] = embedding_2d[:, 1]

# KMeans 分群
kmeans = KMeans(n_clusters=4, random_state=42)
df_unique['cluster'] = kmeans.fit_predict(embedding_2d)

# hover info 加上平均打擊率與情緒
df_unique['hover'] = (
    "Team: " + df_unique['team_name'] +
    "<br>Player: " + df_unique['선수명'] +
    "<br>Sentiment: " + df_unique['custom_sentiment_score'].round(2).astype(str) +
    "<br>Team Avg Sentiment: " + df_unique['avg_sentiment'].round(2).astype(str) +
    "<br>Team Avg BA: " + df_unique['avg_batting_avg'].round(3).astype(str) +
    "<br>Lyrics: " + df_unique['Song_lyrics'].str.slice(0, 40) + "..."
)

# 畫圖（顏色表示隊伍，大小為 team 平均打擊率）
fig = px.scatter(
    df_unique,
    x='umap_x',
    y='umap_y',
    color='team_name',
    size='avg_batting_avg',
    hover_name='선수명',
    hover_data={'umap_x': False, 'umap_y': False, 'cluster': True, 'hover': True},
    title="UMAP: Team Fight Songs vs. Batting Avg & Sentiment",
    size_max=30,
    template='plotly_white'
)

fig.update_traces(marker=dict(line=dict(width=0.5, color='DarkSlateGrey')))
fig.update_layout(legend_title_text='Team', hovermode='closest')
fig.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
import umap
import seaborn as sns

# 字體設定（mac: AppleGothic, Windows: Malgun Gothic）
plt.rcParams['font.family'] = 'AppleGothic'

# 載入資料
df = pd.read_csv("with_sentiment_boost_rounded.csv")

# 過濾：有歌詞且不為"none"
df = df[df['Song_lyrics'].notna() & (df['Song_lyrics'] != "none")].copy()

# 過濾重複（每位球員只保留一首）
df_unique = df.drop_duplicates(subset=["team_name", "선수명", "Song_lyrics"]).copy()

# 取得歌詞與球隊名稱
lyrics = df_unique["Song_lyrics"].tolist()
teams = df_unique["team_name"].tolist()

# 產生嵌入（BERT）
model = SentenceTransformer('distiluse-base-multilingual-cased-v2')
embeddings = model.encode(lyrics, show_progress_bar=True)

# UMAP降維
reducer = umap.UMAP(n_components=2, random_state=42)
embedding_2d = reducer.fit_transform(embeddings)

# 整理 dataframe
df_embed = pd.DataFrame(embedding_2d, columns=["x", "y"])
df_embed["team_name"] = teams
df_embed["선수명"] = df_unique["선수명"].values

# 畫圖（使用 seaborn color palette）
plt.figure(figsize=(12, 8))
sns.scatterplot(
    data=df_embed,
    x="x",
    y="y",
    hue="team_name",
    palette="tab10",
    s=70,
    alpha=0.8
)
plt.title("UMAP of Fight Songs by Team")
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


H11: 情緒分數較高的歌詞出現在打序較前段的打者身上

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy.stats import spearmanr

# 字體設定
plt.rcParams['font.family'] = 'AppleGothic'

# 載入資料
df = pd.read_csv("with_sentiment_boost_rounded.csv")

# 篩選：有情緒、有打序
df = df[df['custom_sentiment_score'].notna() & df['타순'].notna()]

# Spearman 相關係數
corr, pval = spearmanr(df['custom_sentiment_score'], df['타순'])
print(f"Spearman correlation: r = {corr:.4f}, p = {pval:.4f}")

# OLS 回歸：看是否情緒分數越高，打序越小
model = smf.ols("타순 ~ custom_sentiment_score", data=df).fit()
print(model.summary())

# 視覺化
sns.lmplot(
    data=df,
    x="custom_sentiment_score",
    y="타순",
    height=6,
    aspect=1.3,
    scatter_kws={"alpha": 0.3},
    line_kws={"color": "red"}
)
plt.title("Emotion Score vs. Batting Order")
plt.xlabel("Emotion Score")
plt.ylabel("Batting Order (타순)")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 字體設定（Mac 專用）
plt.rcParams['font.family'] = 'AppleGothic'

# 載入資料
df = pd.read_csv("with_sentiment_boost_rounded.csv")

# 篩選：有情緒分數、有打序
df = df[df['custom_sentiment_score'].notna() & df['타순'].notna()]

# 打序轉換為 int（如果是 float）
df['타순'] = df['타순'].astype(int)

# Boxplot：打序 vs. 情緒分數
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x="타순", y="custom_sentiment_score", palette="Set3")
plt.title("Emotion Score by Batting Order")
plt.xlabel("Batting Order (타순)")
plt.ylabel("Emotion Score")
plt.tight_layout()
plt.show()


In [ ]:
# 載入資料
df = pd.read_csv("with_sentiment_boost_rounded.csv")

# 篩選：有情緒分數、有打序
df = df[df['custom_sentiment_score'].notna() & df['타순'].notna()]

# 每位選手：取應援曲情緒分數（只取一首歌即可） + 該選手的平均打序
player_sentiment = df.groupby('선수명')['custom_sentiment_score'].first()
player_batting_order = df.groupby('선수명')['타순'].mean()

# 合併
summary_df = pd.DataFrame({
    'sentiment': player_sentiment,
    'avg_order': player_batting_order
}).dropna()

# 視覺化：打序轉成 int bins（或保持連續數值）
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
sns.boxplot(x=summary_df['avg_order'].round().astype(int), y=summary_df['sentiment'], palette="Set2")
plt.title("Sentiment Score by Avg. Batting Order (Per Player)")
plt.xlabel("Average Batting Order (Rounded)")
plt.ylabel("Emotion Score")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

# 載入資料
df = pd.read_csv("with_sentiment_boost_rounded.csv")

# 篩選有打序與情緒分數
df = df[df['custom_sentiment_score'].notna() & df['타순'].notna()]

# 每位選手的唯一情緒 + 平均打序
player_sentiment = df.groupby('선수명')['custom_sentiment_score'].first()
player_order = df.groupby('선수명')['타순'].mean()

summary_df = pd.DataFrame({
    'sentiment': player_sentiment,
    'avg_order': player_order
}).dropna()

# 線性回歸模型：情緒預測平均打序
model = smf.ols("avg_order ~ sentiment", data=summary_df).fit()
print(model.summary())


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

# 讀取資料
df = pd.read_csv("with_sentiment_boost_rounded.csv")

# 只保留有打序、有歌詞、有情緒分數的資料
df_clean = df[df['custom_sentiment_score'].notna() & df['타순'].notna() & df['Song_lyrics'].notna()]
df_clean = df_clean[df_clean['Song_lyrics'] != "none"]

# 每位選手搭配不同歌詞的唯一組合
df_unique = df_clean.drop_duplicates(subset=['선수명', 'Song_lyrics'])

# 計算平均打序（若某人有多首歌詞，就平均）
summary_df = df_unique.groupby('선수명').agg(
    avg_order=('타순', 'mean'),
    sentiment=('custom_sentiment_score', 'mean')
).reset_index()

# Boxplot：將平均打序四捨五入
summary_df['rounded_order'] = summary_df['avg_order'].round()

plt.figure(figsize=(12, 6))
sns.boxplot(data=summary_df, x='rounded_order', y='sentiment', palette='Paired')
plt.title('Sentiment Score by Avg. Batting Order (Per Player)')
plt.xlabel('Average Batting Order (Rounded)')
plt.ylabel('Emotion Score')
plt.tight_layout()
plt.show()

# 回歸分析：情緒分數是否能預測打序（越前面越低）
model = smf.ols("avg_order ~ sentiment", data=summary_df).fit()
print(model.summary())


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from statsmodels.miscmodels.ordinal_model import OrderedModel
from statsmodels.regression.mixed_linear_model import MixedLM

# Load data
df = pd.read_csv("with_sentiment_boost_rounded.csv")

# Filter: valid at-bats and sentiment score
df = df[(df['타수'] > 0) & df['custom_sentiment_score'].notna()]

# Batting average & numeric order
df['batting_average'] = df['안타'] / df['타수']
df['batting_order'] = pd.to_numeric(df['타순'], errors='coerce')
df = df.dropna(subset=['batting_order'])

# Player-level summary
order_df = (
    df.groupby('선수명')
    .agg(avg_order=('batting_order', 'mean'),
         sentiment=('custom_sentiment_score', 'mean'),
         team=('team_name', 'first'))
    .reset_index()
)

# Binning batting order into categories
def order_category(x):
    if x <= 3:
        return 'front'
    elif x <= 6:
        return 'middle'
    else:
        return 'back'

order_df['order_cat'] = order_df['avg_order'].apply(order_category)

# ------------------------------
# 1. Ordinal Logistic Regression
# ------------------------------
order_df['order_cat'] = pd.Categorical(order_df['order_cat'], categories=['front', 'middle', 'back'], ordered=True)
ordinal_model = OrderedModel(
    order_df['order_cat'],
    order_df[['sentiment']],
    distr='logit'
)
ordinal_result = ordinal_model.fit(method='bfgs')
print("\n=== Ordinal Logistic Regression ===")
print(ordinal_result.summary())

# ------------------------------
# 2. Mixed Effects Regression
# ------------------------------
scaler = StandardScaler()
order_df['z_sentiment'] = scaler.fit_transform(order_df[['sentiment']])

mixed_model = MixedLM.from_formula('avg_order ~ z_sentiment', groups='team', data=order_df)
mixed_result = mixed_model.fit(reml=False)
print("\n=== Mixed Effects Model (Team Random Effect) ===")
print(mixed_result.summary())


In [ ]:
import pandas as pd

df = pd.read_csv("with_sentiment_boost_rounded.csv")
df = df[df['Song_lyrics'].notna() & (df['Song_lyrics'] != "none") & df['타순'].notna()]

# 將打序轉為類別
def order_to_cat(x):
    if x <= 3:
        return "front"
    elif x <= 6:
        return "middle"
    else:
        return "back"

df['order_cat'] = df['타순'].astype(int).apply(order_to_cat)


In [ ]:
from transformers import BertTokenizer, BertModel
import torch

tokenizer = BertTokenizer.from_pretrained("klue/bert-base")
model = BertModel.from_pretrained("klue/bert-base", output_attentions=False)
model.eval()

def get_embeddings(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128, padding="max_length")
    with torch.no_grad():
        outputs = model(**inputs)
    # 取每個詞的輸出向量 (sequence_output) [batch_size, seq_len, hidden_size]
    return inputs, outputs.last_hidden_state.squeeze(0)  # (seq_len, hidden)


new way to use sharp (top 10)

In [ ]:
import pandas as pd

# 載入歌詞資料
df = pd.read_csv("with_sentiment_boost_rounded.csv")

# 移除沒有歌詞的
df = df[df['Song_lyrics'].notna() & (df['Song_lyrics'] != "none")]

# 每人保留一首（取最大情緒分數）
df = (
    df.sort_values('custom_sentiment_score', ascending=False)
      .drop_duplicates(subset=['선수명'])
)

# 取前10首
top10_df = df.head(10).copy()
lyrics = top10_df['Song_lyrics'].tolist()


In [ ]:
from transformers import BertTokenizer, BertModel
import torch

# 載入 KoBERT 或其他適用的 BERT 模型
tokenizer = BertTokenizer.from_pretrained('klue/bert-base')
model = BertModel.from_pretrained('klue/bert-base')
model.eval()

# 取得 CLS 向量（整首歌詞的代表性向量）
embeddings = []
tokens_list = []

for text in lyrics:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    tokens_list.append(tokenizer.convert_ids_to_tokens(inputs['input_ids'][0]))
    with torch.no_grad():
        outputs = model(**inputs)
        cls_embedding = outputs.last_hidden_state[0, 0, :].numpy()  # CLS token
        embeddings.append(cls_embedding)


In [ ]:
import shap
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
import numpy as np

# 假設前3位為前段打序，其餘為後段（你可根據實際打序自定義）
labels = [1 if i < 4 else 0 for i in range(len(embeddings))]  # 1=前段, 0=後段

X = np.stack(embeddings)
y = np.array(labels)

# 建立模型
model = LogisticRegression().fit(X, y)

# 建立 explainer 並計算 SHAP 值
explainer = shap.Explainer(model, X)
shap_values = explainer(X)

# 顯示 summary plot
shap.plots.beeswarm(shap_values)


了解「哪些詞語」最能預測打序為前段。

In [ ]:
import pandas as pd

# 載入資料
df = pd.read_csv("with_sentiment_boost_rounded.csv")
df = df[df['Song_lyrics'].notna() & (df['Song_lyrics'] != "none")]

# 計算每位球員的平均打序
avg_order = df.groupby('선수명')['타순'].mean().reset_index()

# 根據平均打序分類為前段/後段（前 4 打席為前段）
avg_order['order_label'] = avg_order['타순'].apply(lambda x: 1 if x <= 4 else 0)

# 合併回原始資料框
df = pd.merge(df, avg_order[['선수명', 'order_label']], on='선수명', how='left')

# 移除重複記錄（保留情緒分數最高的）
df = df.sort_values('custom_sentiment_score', ascending=False)
df = df.drop_duplicates(subset=['선수명'])

texts = df['Song_lyrics'].tolist()
labels = df['order_label'].tolist()


In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
import torch

tokenizer = BertTokenizer.from_pretrained("klue/bert-base")

class LyricsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_len)
        self.labels = labels

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

dataset = LyricsDataset(texts, labels, tokenizer)
loader = DataLoader(dataset, batch_size=8, shuffle=True)


In [ ]:
from torch.optim import AdamW
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BertForSequenceClassification.from_pretrained("klue/bert-base", num_labels=2).to(device)
optimizer = AdamW(model.parameters(), lr=5e-5)

model.train()
for epoch in range(3):
    for batch in tqdm(loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()


In [ ]:
from transformers_interpret import SequenceClassificationExplainer

explainer = SequenceClassificationExplainer(model, tokenizer)
sample_text = texts[3]  # 換成你想分析的句子

word_attributions = explainer(sample_text)
explainer.visualize()
